# Evaluación de las respuestas del RAG

En este cuaderno, vamos a proceder a hacer la evaluación del RAG. Para ello, vamos a utilizar los .jsonl generados en el cuaderno previo. En estos .jsonl, tenemos, entre otras cosas:
- La pregunta
- La respuesta golden
- La respuesta del RAG
- El capítulo en el que aparece la respuesta golden
- Los chunks recuperados para contestas
- El modelo que ha respondido a la pregunta

Usando estos datos, vamos a evaluar qué tan bueno es el RAG, y como cambia la eficacia de este al cambia el modelo generado de respuestas utilizando siempre la misma base de datos y el mismo modelo de embeddings. 

La comparación entre modelo se hace para, de alguna forma, evaluar también que tan buena ha sido la creación de la base datos. No todo el mundo tiene disponible una GPU A100 con 80 GB de VRAM. Por eso mismo, hay que intentar "optimizar" lo más posible el RAG, probando modelos un poco más sencillos y más ligeros. Si estos modelos son capaces de responder razonablemente bien, no sería necesario utilizar modelos tan grandes ni tener equipos tan costosos.

Para evaluarlo, vamos a empezar por medir la distancia coseno de la respuesta golden y de la respuesta del RAG:

- Si la distancia es menor a 0.35, daremos la respuesta por mala
- Si la distancia está entre 0.35 y 0.4, entrará en la categoría borderline. En este caso, utilizaremos cross-encoder para ver si la respuesta es correcta o no
- Si la distancia es superiro a 0.4, la respuesta se considerará como positiva.


Estos umbrales se han elegido basándonos en la experiencia. Habiendo hecho pruebas a mano con anterioridad, hemos decidido que estos serán los umbrales.

### Aspectos a tener en cuenta:
- Para medir la distancia, utilizaremos toda la respuesta del RAG. Hay veces que la respuesta tiene "ruido": razonamientos del modelo, fallos... Como queremos evalaur que tan bien responde, vemos combeniente utilizar toda la respuesta, aunque tenga bugs.
- Se han añadido 5 preguntas trampa. Es decir, preguntas que no pueden responderse con el libro porque son preguntas directamente incorrectas. Así podremos ver hasta qué punto el modelo alucina o no.

Teniendo esto claro, vamos a proceder con esta primera forma de evaluar:

In [52]:
# =========================
# INSTALL
# =========================
# Ejecuta esta celda solo si no tienes estas librerías instaladas.
!pip install -q -U sentence-transformers transformers accelerate pandas tqdm

In [53]:
# =========================
# IMPORTS
# =========================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from google.colab import drive


In [54]:
# =========================
# GOOGLE DRIVE
# =========================

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
# =========================
# CONFIG
# =========================

# Carpeta donde están tus 5 jsonl
JSONL_DIR = Path("/content/drive/MyDrive/NLP_PRACTICA/evaluacion_rag_runs/q&a_fixed")

# Carpeta donde guardar resultados
OUTPUT_DIR = Path("/content/drive/MyDrive/NLP_PRACTICA/evaluation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Extensión de entrada
JSONL_PATTERN = "*.jsonl"

# Modelo para embeddings de respuestas
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"

# Cross-encoder para casos dudosos
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L12-v2"

# Umbrales que has definido
COSINE_BAD_THRESHOLD = 0.35
COSINE_GOOD_THRESHOLD = 0.40

# Umbral del cross-encoder para decidir si una respuesta dudosa es correcta.
# Este valor es calibrable. Lo dejamos explícito para poder ajustarlo después.
CROSS_ENCODER_THRESHOLD = 0.01

# Batch sizes
EMBEDDING_BATCH_SIZE = 16
CROSS_ENCODER_BATCH_SIZE = 16

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


# =========================
# FAITHFULNESS CONFIG
# =========================

FAITHFULNESS_CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L12-v2"

LETTUCE_MODEL_NAME = "KRLabsOrg/lettucedect-210m-eurobert-es-v1"

FAITH_COSINE_THRESHOLD = 0.35
FAITH_CROSS_ENCODER_THRESHOLD = 0.40

LETTUCE_MAX_CONTEXT_CHARS = 12000

LETTUCE_HALLUCINATION_RATIO_THRESHOLD = 0.35





Device: cuda


In [56]:
# =========================
# LOAD JSONL FILES
# =========================

def load_jsonl_file(path: Path):
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                row = json.loads(line)
                row["_source_file"] = path.name
                row["_line_num"] = line_num
                rows.append(row)

            except json.JSONDecodeError as e:
                print(f"Error leyendo {path.name}, línea {line_num}: {e}")

    return rows


jsonl_files = sorted(JSONL_DIR.glob(JSONL_PATTERN))

print("Archivos encontrados:")
for p in jsonl_files:
    print(" -", p.name)

all_rows = []

for path in jsonl_files:
    all_rows.extend(load_jsonl_file(path))

df_eval = pd.DataFrame(all_rows)

print("Filas cargadas:", len(df_eval))
print("Columnas:", list(df_eval.columns))

display(df_eval.head())

Archivos encontrados:
 - gemma3_4b_4bit_fixed.jsonl
 - gemma3_4b_bf16_fixed.jsonl
 - gemma4_31b_4bit_fixed.jsonl
 - gemma4_e4b_4bit_fixed.jsonl
 - gemma4_e4b_bf16_fixed.jsonl
Filas cargadas: 525
Columnas: ['run_id', 'model_alias', 'model_id', 'quant', 'idx', 'question', 'gold_answer', 'gold_chapters', 'retrieved_chunks', 'generated_answer', 'latency_sec', 'top_k', 'is_hard_negative', 'max_new_tokens', '_source_file', '_line_num']


run_id     model_alias              model_id quant  idx  \
0  20260429134834  gemma3_4b_4bit  google/gemma-3-4b-it  4bit    1   
1  20260429134834  gemma3_4b_4bit  google/gemma-3-4b-it  4bit    2   
2  20260429134834  gemma3_4b_4bit  google/gemma-3-4b-it  4bit    3   
3  20260429134834  gemma3_4b_4bit  google/gemma-3-4b-it  4bit    4   
4  20260429134834  gemma3_4b_4bit  google/gemma-3-4b-it  4bit    5   

                                                                            question  \
0  ¿Quién lidera la partida de exploradores de la Guardia de la Noche en el prólogo?   
1       ¿Cómo se llaman los otros dos exploradores que acompañan a Ser Waymar Royce?   
2              ¿Qué criaturas aparecen más allá del Muro y matan a Ser Waymar Royce?   
3                                        ¿Qué le ocurre a Will al final del prólogo?   
4                     ¿Quién ejecuta a Gared por desertar de la Guardia de la Noche?   

                                                gold_answer gold_chapters  \
0                                         Ser Waymar Royce.     [Prólogo]   
1                                             Will y Gared.     [Prólogo]   
2                                                Los Otros.     [Prólogo]   
3  Es atacado por el cadáver reanimado de Ser Waymar Royce.     [Prólogo]   
4                                             Eddard Stark.      [Bran I]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [57]:
# =========================
# CREATE retrieved_chunks_simple
# =========================

def simplify_chunk(chunk):
    if not isinstance(chunk, dict):
        return {}

    metadata = chunk.get("metadata", {})

    return {
        "rank": chunk.get("rank"),
        "rank_global": chunk.get("rank_global"),
        "score": chunk.get("score"),
        "source": chunk.get("source"),
        "chapter_title": metadata.get("chapter_title") if isinstance(metadata, dict) else None,
        "chapter": metadata.get("chapter") if isinstance(metadata, dict) else None,
        "chapter_order": metadata.get("chapter_order") if isinstance(metadata, dict) else None,
        "chapter_id": metadata.get("chapter_id") if isinstance(metadata, dict) else None,
        "pov": metadata.get("pov") if isinstance(metadata, dict) else None,
        "chunk_index": metadata.get("chunk_index") if isinstance(metadata, dict) else None,
        "main_event": metadata.get("main_event") if isinstance(metadata, dict) else None,
    }


def simplify_chunks(chunks):
    if not isinstance(chunks, list):
        return []

    return [simplify_chunk(chunk) for chunk in chunks]


df_eval["retrieved_chunks_simple"] = df_eval["retrieved_chunks"].apply(simplify_chunks)



In [58]:
# Mostrar TODAS las columnas
pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', None)

pd.set_option('display.max_colwidth', None)

pd.set_option('display.width', None)

pd.set_option('display.float_format', '{:.6f}'.format)

In [59]:
# =========================
# BASIC DERIVED COLUMNS
# =========================

def safe_len(x):
    if isinstance(x, list):
        return len(x)
    return 0


def get_top1_field(chunks, field, default=None):
    if not isinstance(chunks, list) or len(chunks) == 0:
        return default

    return chunks[0].get(field, default)


df_eval["num_retrieved_chunks"] = df_eval["retrieved_chunks"].apply(safe_len)

df_eval["top1_source"] = df_eval["retrieved_chunks"].apply(
    lambda x: get_top1_field(x, "source")
)

df_eval["top1_score"] = df_eval["retrieved_chunks"].apply(
    lambda x: get_top1_field(x, "score")
)

df_eval["top1_rank_global"] = df_eval["retrieved_chunks"].apply(
    lambda x: get_top1_field(x, "rank_global")
)

# Aseguramos strings sin limpiar contenido.
# Solo evitamos NaN para que los modelos no fallen.
df_eval["gold_answer"] = df_eval["gold_answer"].fillna("").astype(str)
df_eval["generated_answer"] = df_eval["generated_answer"].fillna("").astype(str)

display(df_eval[
    [
        "model_alias",
        "idx",
        "question",
        "gold_answer",
        "generated_answer",
        "num_retrieved_chunks",
        "top1_source",
        "top1_score"
    ]
].head())

,model_alias,idx,question,gold_answer,generated_answer,num_retrieved_chunks,top1_source,top1_score
0,gemma3_4b_4bit,1,¿Quién lidera la partida de exploradores de la Guardia de la Noche en el prólogo?,Ser Waymar Royce.,Jon.,5,libro,0.305916
1,gemma3_4b_4bit,2,¿Cómo se llaman los otros dos exploradores que acompañan a Ser Waymar Royce?,Will y Gared.,No lo sé con los fragmentos proporcionados.,5,resumen,0.314942
2,gemma3_4b_4bit,3,¿Qué criaturas aparecen más allá del Muro y matan a Ser Waymar Royce?,Los Otros.,Otros.,5,resumen,0.288064
3,gemma3_4b_4bit,4,¿Qué le ocurre a Will al final del prólogo?,Es atacado por el cadáver reanimado de Ser Waymar Royce.,El Fragmento 1 describe que el cadáver de Royce comienza a asfixiarle hasta la muerte. visualización,5,resumen,0.111185
4,gemma3_4b_4bit,5,¿Quién ejecuta a Gared por desertar de la Guardia de la Noche?,Eddard Stark.,Eddard Stark.,5,libro,0.333214


In [60]:
# =========================
# COSINE SIMILARITY EVALUATION
# =========================

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device
)

gold_texts = df_eval["gold_answer"].tolist()
generated_texts = df_eval["generated_answer"].tolist()

gold_embeddings = embedding_model.encode(
    gold_texts,
    batch_size=EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

generated_embeddings = embedding_model.encode(
    generated_texts,
    batch_size=EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Como los embeddings están normalizados, el producto escalar equivale a cosine similarity.
cosine_scores = np.sum(gold_embeddings * generated_embeddings, axis=1)

df_eval["answer_cosine_similarity"] = cosine_scores


def classify_cosine(score):
    if score < COSINE_BAD_THRESHOLD:
        return "bad"
    elif score <= COSINE_GOOD_THRESHOLD:
        return "borderline"
    else:
        return "good"


df_eval["cosine_decision"] = df_eval["answer_cosine_similarity"].apply(classify_cosine)
df_eval["needs_cross_encoder"] = df_eval["cosine_decision"].eq("borderline")

display(df_eval[
    [
        "model_alias",
        "idx",
        "gold_answer",
        "generated_answer",
        "answer_cosine_similarity",
        "cosine_decision",
        "needs_cross_encoder"
    ]
].head(20))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

,model_alias,idx,gold_answer,generated_answer,answer_cosine_similarity,cosine_decision,needs_cross_encoder
0,gemma3_4b_4bit,1,Ser Waymar Royce.,Jon.,0.467441,good,False
1,gemma3_4b_4bit,2,Will y Gared.,No lo sé con los fragmentos proporcionados.,0.328746,bad,False
2,gemma3_4b_4bit,3,Los Otros.,Otros.,0.740967,good,False
3,gemma3_4b_4bit,4,Es atacado por el cadáver reanimado de Ser Waymar Royce.,El Fragmento 1 describe que el cadáver de Royce comienza a asfixiarle hasta la muerte. visualización,0.506943,good,False
4,gemma3_4b_4bit,5,Eddard Stark.,Eddard Stark.,1.005821,good,False
5,gemma3_4b_4bit,6,Una loba huargo y un venado.,Un lobo muerto y un carnicero.,0.593859,good,False
6,gemma3_4b_4bit,7,Seis: cinco primero y después el cachorro albino de Jon.,No lo sé con los fragmentos proporcionados.,0.193310,bad,False
7,gemma3_4b_4bit,8,Que Jon Arryn ha muerto.,La carta indica que Jon Arryn fue asesinado por los Lannister.cppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcpp,0.705770,good,False
8,gemma3_4b_4bit,9,Mano del Rey.,Mano del Rey.ัศน์,0.832964,good,False
9,gemma3_4b_4bit,10,En las criptas de Invernalia.,No lo sé con los fragmentos proporcionados.,0.294827,bad,False


In [61]:
# =========================
# COSINE SUMMARY
# =========================

cosine_summary = (
    df_eval
    .groupby("model_alias")
    .agg(
        n=("idx", "count"),
        mean_cosine=("answer_cosine_similarity", "mean"),
        median_cosine=("answer_cosine_similarity", "median"),
        min_cosine=("answer_cosine_similarity", "min"),
        max_cosine=("answer_cosine_similarity", "max"),
        good_by_cosine=("cosine_decision", lambda x: (x == "good").sum()),
        borderline_by_cosine=("cosine_decision", lambda x: (x == "borderline").sum()),
        bad_by_cosine=("cosine_decision", lambda x: (x == "bad").sum()),
    )
    .reset_index()
)

cosine_summary["good_by_cosine_rate"] = cosine_summary["good_by_cosine"] / cosine_summary["n"]
cosine_summary["borderline_by_cosine_rate"] = cosine_summary["borderline_by_cosine"] / cosine_summary["n"]
cosine_summary["bad_by_cosine_rate"] = cosine_summary["bad_by_cosine"] / cosine_summary["n"]

display(cosine_summary)

,model_alias,n,mean_cosine,median_cosine,min_cosine,max_cosine,good_by_cosine,borderline_by_cosine,bad_by_cosine,good_by_cosine_rate,borderline_by_cosine_rate,bad_by_cosine_rate
0,gemma3_4b_4bit,105,0.552472,0.531909,0.192730,1.005821,73,4,28,0.695238,0.038095,0.266667
1,gemma3_4b_bf16,105,0.562720,0.544389,0.194683,1.006113,77,7,21,0.733333,0.066667,0.200000
2,gemma4_31b_4bit,105,0.548678,0.558496,0.223022,0.805827,93,4,8,0.885714,0.038095,0.076190
3,gemma4_e4b_4bit,105,0.472273,0.463350,0.167258,1.004640,68,15,22,0.647619,0.142857,0.209524
4,gemma4_e4b_bf16,105,0.535990,0.550835,0.156208,1.004897,68,5,32,0.647619,0.047619,0.304762


In [62]:
# =========================
# CROSS-ENCODER ONLY FOR BORDERLINE CASES
# =========================

df_eval["cross_encoder_score"] = np.nan
df_eval["cross_encoder_decision"] = None

borderline_mask = df_eval["needs_cross_encoder"]

num_borderline = int(borderline_mask.sum())
print("Casos borderline:", num_borderline)

if num_borderline > 0:
    cross_encoder = CrossEncoder(
        CROSS_ENCODER_MODEL_NAME,
        device=device
    )

    pairs = list(
        zip(
            df_eval.loc[borderline_mask, "gold_answer"].tolist(),
            df_eval.loc[borderline_mask, "generated_answer"].tolist()
        )
    )

    ce_scores = cross_encoder.predict(
        pairs,
        batch_size=CROSS_ENCODER_BATCH_SIZE,
        show_progress_bar=True
    )

    df_eval.loc[borderline_mask, "cross_encoder_score"] = ce_scores

    df_eval.loc[borderline_mask, "cross_encoder_decision"] = np.where(
        df_eval.loc[borderline_mask, "cross_encoder_score"] >= CROSS_ENCODER_THRESHOLD,
        "good",
        "bad"
    )

display(df_eval[
    [
        "model_alias",
        "idx",
        "answer_cosine_similarity",
        "cosine_decision",
        "needs_cross_encoder",
        "cross_encoder_score",
        "cross_encoder_decision",
        "gold_answer",
        "generated_answer"
    ]
].head(30))

Casos borderline: 35


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

,model_alias,idx,answer_cosine_similarity,cosine_decision,needs_cross_encoder,cross_encoder_score,cross_encoder_decision,gold_answer,generated_answer
0,gemma3_4b_4bit,1,0.467441,good,False,NaN,None,Ser Waymar Royce.,Jon.
1,gemma3_4b_4bit,2,0.328746,bad,False,NaN,None,Will y Gared.,No lo sé con los fragmentos proporcionados.
2,gemma3_4b_4bit,3,0.740967,good,False,NaN,None,Los Otros.,Otros.
3,gemma3_4b_4bit,4,0.506943,good,False,NaN,None,Es atacado por el cadáver reanimado de Ser Waymar Royce.,El Fragmento 1 describe que el cadáver de Royce comienza a asfixiarle hasta la muerte. visualización
4,gemma3_4b_4bit,5,1.005821,good,False,NaN,None,Eddard Stark.,Eddard Stark.
5,gemma3_4b_4bit,6,0.593859,good,False,NaN,None,Una loba huargo y un venado.,Un lobo muerto y un carnicero.
6,gemma3_4b_4bit,7,0.193310,bad,False,NaN,None,Seis: cinco primero y después el cachorro albino de Jon.,No lo sé con los fragmentos proporcionados.
7,gemma3_4b_4bit,8,0.705770,good,False,NaN,None,Que Jon Arryn ha muerto.,La carta indica que Jon Arryn fue asesinado por los Lannister.cppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcpp
8,gemma3_4b_4bit,9,0.832964,good,False,NaN,None,Mano del Rey.,Mano del Rey.ัศน์
9,gemma3_4b_4bit,10,0.294827,bad,False,NaN,None,En las criptas de Invernalia.,No lo sé con los fragmentos proporcionados.


In [63]:
# =========================
# FINAL GENERATION DECISION
# =========================

def final_generation_decision(row):
    if row["cosine_decision"] == "bad":
        return "bad"

    if row["cosine_decision"] == "good":
        return "good"

    if row["cosine_decision"] == "borderline":
        return row["cross_encoder_decision"]

    return None


df_eval["generation_eval_decision"] = df_eval.apply(final_generation_decision, axis=1)

df_eval["generation_eval_correct"] = df_eval["generation_eval_decision"].eq("good").astype(int)

display(df_eval[
    [
        "model_alias",
        "idx",
        "question",
        "gold_answer",
        "generated_answer",
        "answer_cosine_similarity",
        "cosine_decision",
        "cross_encoder_score",
        "cross_encoder_decision",
        "generation_eval_decision",
        "generation_eval_correct"
    ]
].head(30))

,model_alias,idx,question,gold_answer,generated_answer,answer_cosine_similarity,cosine_decision,cross_encoder_score,cross_encoder_decision,generation_eval_decision,generation_eval_correct
0,gemma3_4b_4bit,1,¿Quién lidera la partida de exploradores de la Guardia de la Noche en el prólogo?,Ser Waymar Royce.,Jon.,0.467441,good,NaN,None,good,1
1,gemma3_4b_4bit,2,¿Cómo se llaman los otros dos exploradores que acompañan a Ser Waymar Royce?,Will y Gared.,No lo sé con los fragmentos proporcionados.,0.328746,bad,NaN,None,bad,0
2,gemma3_4b_4bit,3,¿Qué criaturas aparecen más allá del Muro y matan a Ser Waymar Royce?,Los Otros.,Otros.,0.740967,good,NaN,None,good,1
3,gemma3_4b_4bit,4,¿Qué le ocurre a Will al final del prólogo?,Es atacado por el cadáver reanimado de Ser Waymar Royce.,El Fragmento 1 describe que el cadáver de Royce comienza a asfixiarle hasta la muerte. visualización,0.506943,good,NaN,None,good,1
4,gemma3_4b_4bit,5,¿Quién ejecuta a Gared por desertar de la Guardia de la Noche?,Eddard Stark.,Eddard Stark.,1.005821,good,NaN,None,good,1
5,gemma3_4b_4bit,6,¿Qué animales muertos encuentran los Stark cerca de los cachorros de huargo?,Una loba huargo y un venado.,Un lobo muerto y un carnicero.,0.593859,good,NaN,None,good,1
6,gemma3_4b_4bit,7,¿Cuántos cachorros de huargo encuentran finalmente los Stark?,Seis: cinco primero y después el cachorro albino de Jon.,No lo sé con los fragmentos proporcionados.,0.193310,bad,NaN,None,bad,0
7,gemma3_4b_4bit,8,¿Qué noticia recibe Ned Stark sobre Jon Arryn?,Que Jon Arryn ha muerto.,La carta indica que Jon Arryn fue asesinado por los Lannister.cppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcpp,0.705770,good,NaN,None,good,1
8,gemma3_4b_4bit,9,¿Qué cargo le ofrece Robert Baratheon a Ned Stark?,Mano del Rey.,Mano del Rey.ัศน์,0.832964,good,NaN,None,good,1
9,gemma3_4b_4bit,10,¿Dónde hablan Robert y Ned sobre Lyanna Stark?,En las criptas de Invernalia.,No lo sé con los fragmentos proporcionados.,0.294827,bad,NaN,None,bad,0


In [64]:
# =========================
# FINAL SUMMARY BY MODEL
# =========================

generation_summary = (
    df_eval
    .groupby(["model_alias", "model_id", "quant"], dropna=False)
    .agg(
        n=("idx", "count"),
        correct=("generation_eval_correct", "sum"),
        accuracy=("generation_eval_correct", "mean"),
        mean_cosine=("answer_cosine_similarity", "mean"),
        median_cosine=("answer_cosine_similarity", "median"),
        bad_by_cosine=("cosine_decision", lambda x: (x == "bad").sum()),
        borderline_by_cosine=("cosine_decision", lambda x: (x == "borderline").sum()),
        good_by_cosine=("cosine_decision", lambda x: (x == "good").sum()),
        cross_encoder_used=("needs_cross_encoder", "sum"),
        mean_cross_encoder_score=("cross_encoder_score", "mean"),
        mean_latency_sec=("latency_sec", "mean"),
        median_latency_sec=("latency_sec", "median"),
    )
    .reset_index()
)

generation_summary["bad_by_cosine_rate"] = generation_summary["bad_by_cosine"] / generation_summary["n"]
generation_summary["borderline_by_cosine_rate"] = generation_summary["borderline_by_cosine"] / generation_summary["n"]
generation_summary["good_by_cosine_rate"] = generation_summary["good_by_cosine"] / generation_summary["n"]
generation_summary["cross_encoder_used_rate"] = generation_summary["cross_encoder_used"] / generation_summary["n"]

generation_summary = generation_summary.sort_values(
    by=["accuracy", "mean_cosine"],
    ascending=[False, False]
)

display(generation_summary)

,model_alias,model_id,quant,n,correct,accuracy,mean_cosine,median_cosine,bad_by_cosine,borderline_by_cosine,good_by_cosine,cross_encoder_used,mean_cross_encoder_score,mean_latency_sec,median_latency_sec,bad_by_cosine_rate,borderline_by_cosine_rate,good_by_cosine_rate,cross_encoder_used_rate
2,gemma4_31b_4bit,google/gemma-4-31b-it,4bit,105,93,0.885714,0.548678,0.558496,8,4,93,4,-8.984063,23.874762,24.355000,0.076190,0.038095,0.885714,0.038095
1,gemma3_4b_bf16,google/gemma-3-4b-it,bf16,105,77,0.733333,0.562720,0.544389,21,7,77,7,-10.166712,6.514552,6.520000,0.200000,0.066667,0.733333,0.066667
0,gemma3_4b_4bit,google/gemma-3-4b-it,4bit,105,73,0.695238,0.552472,0.531909,28,4,73,4,-9.400354,9.861505,10.017000,0.266667,0.038095,0.695238,0.038095
3,gemma4_e4b_4bit,google/gemma-4-E4B-it,4bit,105,72,0.685714,0.472273,0.463350,22,15,68,15,-6.539347,12.898752,15.115000,0.209524,0.142857,0.647619,0.142857
4,gemma4_e4b_bf16,google/gemma-4-E4B-it,bf16,105,69,0.657143,0.535990,0.550835,32,5,68,5,-3.125892,8.377571,9.794000,0.304762,0.047619,0.647619,0.047619


In [65]:
# =========================
# SAVE RESULTS
# =========================

detailed_csv_path = OUTPUT_DIR / "generation_eval_detailed.csv"
detailed_jsonl_path = OUTPUT_DIR / "generation_eval_detailed.jsonl"
summary_csv_path = OUTPUT_DIR / "generation_eval_summary.csv"

# Para CSV, convertimos retrieved_chunks a string JSON para que no se rompa.
df_eval_csv = df_eval.copy()
df_eval_csv["retrieved_chunks"] = df_eval_csv["retrieved_chunks"].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
)

if "gold_chapters" in df_eval_csv.columns:
    df_eval_csv["gold_chapters"] = df_eval_csv["gold_chapters"].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
    )

df_eval_csv.to_csv(detailed_csv_path, index=False, encoding="utf-8")

generation_summary.to_csv(summary_csv_path, index=False, encoding="utf-8")

with detailed_jsonl_path.open("w", encoding="utf-8") as f:
    for row in df_eval.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

print("Guardado:")
print(" -", detailed_csv_path)
print(" -", detailed_jsonl_path)
print(" -", summary_csv_path)

Guardado:
 - /content/drive/MyDrive/NLP_PRACTICA/evaluation_outputs/generation_eval_detailed.csv
 - /content/drive/MyDrive/NLP_PRACTICA/evaluation_outputs/generation_eval_detailed.jsonl
 - /content/drive/MyDrive/NLP_PRACTICA/evaluation_outputs/generation_eval_summary.csv


Una vez hecha esta primera forma de evaluación, podemos empezar a sacar algunas conclusiones:

- Gemma-4 31B 4-bit:
    - Vemos que, sin lugar a dudas, el mejor modelo es el gemma-4 de 31B de parámetros con cuantización de 4-bits. Esto era esperable, pues es el modelo de Google más reciente de todos. 
    - Este modelo ha acertado 93 de las 105 preguntas, con una precisión del 88.6%. Es una tasa de aciertos realmente buena. 
    - Además, la distancia coseno media es de 0.54, también buena teniendo en cuenta lo comentado previamente de que hay errores en la generación de las respuestas. 
    - Por otro lado, hemos tenido que utilizar el cross-encoder 4 veces, y todas han caido en el saco de las incorrectas.
    - También vemos que es el modelo que más tiempo necesita para contestar, con una media de 24 segundos. Es realmente lento. Si ChatGPT tardase para preguntas tan simples 24 segundos en responder, la gente usaría Google.
- Gemma-3 4B bf16:
    - El siguiente modelo que mejores resultados ha obtenido es el gemma-4 de 4B de parámetros con la cuantización de bf16 (es decir, sin compresión). En esta ocasión, ha acertado 77 de 105 preguntas, con una precisión del 73%. Nada mal teniendo en cuenta que es un modelo con 27B de parámetros menos que el Gemma-4 (aunque esto no es así literalmente, pues el gemma-4 estaba cuantizado).
    - Esta vez, la media de la distancia es superiro, con 0.56 de media.
    - Hemos tenido que utilizar el cross 7 veces, de la cuales todas han sido calificadas como malas.
    - El tiempo medio en responder ha sido el más bajo, con 6.5 segundo, 4 veces más rápido que el gemma-4.

- Gemma-3 4B 4-bit:
    - Este modelo ha acertado 73 preguntas, lo que se traduce en una precisión del 0.69%. Aquí el rendimiento empieza a decaer de forma preocupante, pues casi un tercio de las preguntas están mal contestadas. Sinceramente, yo no usaría un modelo con estas características.
    - La distancia coseno es de 0.55 de media, parecida a las anteriores.
    - El cross se ha utilizado 4 veces, y las cuatro ha sido para dar la respuesta como incorrecta.
    - El tiempo de inferencia ha sido de 10 segundos, algo más alto que alterior.

- Gemma-4 E4B 4-bit:
    - En esta ocasión, se han respondido bien 72 pregunta, una menos que antes, lo que le da al modelo una precisión del 68%.
    - La distancia coseno es la peor, de 0.47
    - El cross se ha utilizado 15 veces (lo cual es esperable con una distancia coseno tan baja). De esas 15 veces, 4 respuestas se han calificado como correctas, y el resto como incorrectas.
    - El tiempo de inferencia es de 15 segundos de media, muy alto para los malos resultados que devuelve el modelo

- Gemma-4 E4B bf16:
    - Este modelo ha acertado 69 preguntas, con una precisión del 65%. Realmente pobre. Un resultado realmente malo teniendo en cuenta que se modelo debería ser superior a todos los demás menos al de 31B de parámetros.
    - La distancia es de 0.54 de media
    - El cross se ha utilizado 5 veces y una respuesta ha sido catalogada como correcta.
    - El tiempo de inferencia es de 9.8 segundos

Como vemos, en general, cuanto peor es el modelo, peor es la respeusta y la calidad del RAG. No obstante, esta no es la única evaluación que vamos a llevar a cabo. 

A continuación, vamos a evaluar el modelo, pero esta vez utilizando n-gramas. Si algún n-grama relevante de la respuesta golden aparece en la respuesta del RAG, se considerará como buena. Veámoslo.

## Evaluación por N-gramas / Keyphrases

Esta métrica evalúa si la respuesta generada contiene los conceptos clave de la respuesta correcta.

### Proceso

1. **Extracción desde la gold**
   - Se normaliza el texto (minúsculas, sin tildes, sin puntuación).
   - Se generan unigramas, bigramas y trigramas (todos los posibles, es decir, todas las combinaciones posibles, pero siguiendo el orden de las palabras).
   - Se detectan entidades por mayúsculas (ej: *Ser Waymar Royce*, *Mano del Rey*). Es decir, se coge el texto sin noralizar, se mirar entidades por mayusculas, y con esas entidades, elegimos con qué ngramas quedarnos de los antes generados.
   - Se filtran n-gramas poco informativos (stopwords). Es decir, si un bigrama es [en la], este se elimina.
   - Se seleccionan las mejores expresiones → **keyphrases**.

2. **Matching en la respuesta generada**
   - Se normaliza la respuesta generada.
   - Se comprueba si aparecen las keyphrases (exacto o parcial fuerte).

3. **Métrica**
   - `overlap = keyphrases_matched / keyphrases_gold`

4. **Decisión**
   - 1 keyphrase → debe aparecer → **good**
   - 2–3 keyphrases → ≥50% → **good**
   - >3 keyphrases → ≥60% → **good**
   - En otro caso → **bad**

### Interpretación

- Captura si el modelo mantiene **nombres, entidades y conceptos clave**.
- Complementa al cosine (semántica) con señal **léxica estructural**.

No obstante, este metodo de evaluación no es ni especialmente bueno ni especialmente útil por sí solo. Estará mejor si lo usamos combinado con el resto de metodos de evaluación que veremos más adelante.

In [66]:
import re
import unicodedata
from collections import Counter

# =========================
# NGRAM / KEYPHRASE CONFIG
# =========================

SPANISH_STOPWORDS = {
    "a", "al", "algo", "ante", "antes", "como", "con", "contra", "cual", "cuando",
    "de", "del", "desde", "donde", "durante", "e", "el", "ella", "ellas", "ellos",
    "en", "entre", "era", "eran", "es", "esa", "esas", "ese", "eso", "esos", "esta",
    "estaba", "estaban", "estado", "estan", "estar", "este", "esto", "estos", "fue",
    "fueron", "ha", "habia", "hacia", "hasta", "hay", "la", "las", "le", "les",
    "lo", "los", "mas", "me", "mi", "mientras", "muy", "no", "o", "para", "pero",
    "por", "porque", "que", "quien", "se", "ser", "si", "sin", "sobre", "su", "sus",
    "tambien", "te", "un", "una", "unas", "uno", "unos", "y", "ya"
}

# Palabras funcionales permitidas dentro de expresiones tipo "Mano del Rey"
CONNECTOR_WORDS = {"de", "del", "la", "las", "el", "los", "y"}

NGRAM_GOOD_THRESHOLD = 0.60
NGRAM_MIN_KEYPHRASES = 1

In [67]:
# =========================
# TEXT HELPERS
# =========================

def strip_accents(text):
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text


def normalize_text(text):
    """
    Normalización ligera para comparar:
    - minúsculas
    - sin tildes
    - sin puntuación
    - espacios compactados

    No limpia la respuesta generada a nivel semántico.
    Solo normaliza para poder comparar strings.
    """
    text = "" if text is None else str(text)
    text = strip_accents(text.lower())
    text = re.sub(r"[^a-zA-Z0-9ñÑáéíóúÁÉÍÓÚüÜ\s-]", " ", text)
    text = text.replace("-", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    norm = normalize_text(text)
    if not norm:
        return []
    return norm.split()


def original_tokens(text):
    text = "" if text is None else str(text)
    return re.findall(r"\b[\wÁÉÍÓÚÜÑáéíóúüñ-]+\b", text)


def is_capitalized_token(tok):
    if not tok:
        return False
    return tok[0].isupper() and len(tok) > 1


def generate_ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def informative_ngram(ngram):
    toks = ngram.split()

    if not toks:
        return False

    non_stop = [t for t in toks if t not in SPANISH_STOPWORDS]

    # Descartar si todo son stopwords.
    if len(non_stop) == 0:
        return False

    # Descartar unigramas demasiado cortos.
    if len(toks) == 1 and len(toks[0]) <= 2:
        return False

    # Descartar n-gramas que empiezan o acaban con stopword débil,
    # salvo conectores internos.
    if len(toks) >= 2:
        if toks[0] in SPANISH_STOPWORDS and toks[0] not in CONNECTOR_WORDS:
            return False
        if toks[-1] in SPANISH_STOPWORDS:
            return False

    return True

In [68]:
# =========================
# KEYPHRASE EXTRACTION
# =========================

def extract_capitalized_phrases(text):
    """
    Detecta secuencias de tokens con mayúscula en el texto original.
    Permite conectores internos: de, del, la, el, y...
    
    Ejemplos:
    - Ser Waymar Royce
    - Mano del Rey
    - Guardia de la Noche
    """
    toks = original_tokens(text)
    phrases = []

    i = 0
    while i < len(toks):
        tok = toks[i]

        if is_capitalized_token(tok):
            phrase = [tok]
            j = i + 1

            while j < len(toks):
                next_tok = toks[j]
                next_norm = normalize_text(next_tok)

                if is_capitalized_token(next_tok):
                    phrase.append(next_tok)
                    j += 1

                elif next_norm in CONNECTOR_WORDS and j + 1 < len(toks) and is_capitalized_token(toks[j + 1]):
                    phrase.append(next_tok)
                    phrase.append(toks[j + 1])
                    j += 2

                else:
                    break

            if len(phrase) >= 1:
                phrases.append(" ".join(phrase))

            i = j
        else:
            i += 1

    return phrases


def score_candidate(ngram, capitalized_phrases_norm):
    toks = ngram.split()
    n = len(toks)

    score = 0.0

    # Preferimos bigramas/trigramas cuando existen.
    if n == 3:
        score += 3.0
    elif n == 2:
        score += 2.0
    elif n == 1:
        score += 1.0

    # Premiar si forma parte de una entidad detectada por mayúsculas.
    for phrase in capitalized_phrases_norm:
        if ngram == phrase:
            score += 5.0
        elif ngram in phrase and len(ngram) >= 4:
            score += 2.0

    # Premiar tokens no stopword.
    non_stop = [t for t in toks if t not in SPANISH_STOPWORDS]
    score += len(non_stop) * 0.7

    # Penalizar n-gramas con demasiadas stopwords.
    stop_count = len(toks) - len(non_stop)
    score -= stop_count * 0.3

    # Penalizar cosas demasiado cortas.
    if len(ngram) <= 3:
        score -= 2.0

    return score


def extract_gold_keyphrases(gold_answer, max_phrases=8):
    """
    Extrae keyphrases automáticamente desde la respuesta correcta.
    No usa diccionario de dominio.
    """
    gold_answer = "" if gold_answer is None else str(gold_answer)

    tokens = tokenize(gold_answer)
    if not tokens:
        return []

    cap_phrases = extract_capitalized_phrases(gold_answer)
    cap_phrases_norm = [normalize_text(p) for p in cap_phrases if normalize_text(p)]

    candidates = []

    # Entidades por mayúsculas como candidatos fuertes.
    candidates.extend(cap_phrases_norm)

    # N-gramas de la gold.
    for n in [3, 2, 1]:
        candidates.extend(generate_ngrams(tokens, n))

    # Filtrar candidatos.
    candidates = [
        c for c in candidates
        if informative_ngram(c)
    ]

    # Deduplicar conservando el mejor score.
    scored = {}
    for cand in candidates:
        score = score_candidate(cand, cap_phrases_norm)
        if cand not in scored or score > scored[cand]:
            scored[cand] = score

    # Ordenar por score descendente y longitud descendente.
    ordered = sorted(
        scored.items(),
        key=lambda x: (x[1], len(x[0].split()), len(x[0])),
        reverse=True
    )

    # Evitar redundancia extrema:
    # si ya tenemos "ser waymar royce", no necesitamos "ser waymar"
    selected = []
    for cand, score in ordered:
        is_redundant = False

        for prev in selected:
            if cand in prev and len(cand.split()) < len(prev.split()):
                is_redundant = True
                break

        if not is_redundant:
            selected.append(cand)

        if len(selected) >= max_phrases:
            break

    return selected

In [69]:
# =========================
# KEYPHRASE MATCHING
# =========================

def keyphrase_found(keyphrase, generated_text):
    gen_norm = normalize_text(generated_text)

    if not keyphrase:
        return False

    # Match exacto por frase normalizada.
    pattern = r"\b" + re.escape(keyphrase) + r"\b"
    if re.search(pattern, gen_norm):
        return True

    # Para expresiones largas, aceptar match parcial fuerte.
    toks = keyphrase.split()
    non_stop = [t for t in toks if t not in SPANISH_STOPWORDS]

    if len(non_stop) >= 2:
        matched = sum(1 for t in non_stop if re.search(r"\b" + re.escape(t) + r"\b", gen_norm))
        return matched / len(non_stop) >= 0.80

    return False


def evaluate_ngram_row(row):
    gold = row["gold_answer"]
    generated = row["generated_answer"]

    keyphrases = extract_gold_keyphrases(gold)
    matched = []
    missing = []

    for kp in keyphrases:
        if keyphrase_found(kp, generated):
            matched.append(kp)
        else:
            missing.append(kp)

    total = len(keyphrases)
    match_count = len(matched)

    if total == 0:
        ratio = np.nan
        decision = "unknown"

    else:
        ratio = match_count / total

        # Regla para gold answers muy cortas.
        # Si solo hay una keyphrase, tiene que aparecer.
        if total == 1:
            decision = "good" if match_count == 1 else "bad"

        # Si hay 2 o 3, aceptamos >= 50%.
        elif total <= 3:
            decision = "good" if ratio >= 0.50 else "bad"

        # Si hay más, usamos umbral general.
        else:
            decision = "good" if ratio >= NGRAM_GOOD_THRESHOLD else "bad"

    return pd.Series({
        "gold_keyphrases": keyphrases,
        "matched_keyphrases": matched,
        "missing_keyphrases": missing,
        "ngram_match_count": match_count,
        "ngram_total_gold": total,
        "ngram_overlap_ratio": ratio,
        "ngram_eval_decision": decision,
        "ngram_eval_correct": 1 if decision == "good" else 0 if decision == "bad" else np.nan
    })

In [70]:
# =========================
# APPLY NGRAM / KEYPHRASE EVALUATION
# =========================

ngram_results = df_eval.apply(evaluate_ngram_row, axis=1)

# Añadimos las columnas al MISMO dataframe df_eval
for col in ngram_results.columns:
    df_eval[col] = ngram_results[col]

display(df_eval[
    [
        "model_alias",
        "idx",
        "question",
        "gold_answer",
        "generated_answer",
        "gold_keyphrases",
        "matched_keyphrases",
        "missing_keyphrases",
        "ngram_overlap_ratio",
        "ngram_eval_decision",
        "ngram_eval_correct"
    ]
].head(30))

,model_alias,idx,question,gold_answer,generated_answer,gold_keyphrases,matched_keyphrases,missing_keyphrases,ngram_overlap_ratio,ngram_eval_decision,ngram_eval_correct
0,gemma3_4b_4bit,1,¿Quién lidera la partida de exploradores de la Guardia de la Noche en el prólogo?,Ser Waymar Royce.,Jon.,[waymar royce],[],[waymar royce],0.000000,bad,0
1,gemma3_4b_4bit,2,¿Cómo se llaman los otros dos exploradores que acompañan a Ser Waymar Royce?,Will y Gared.,No lo sé con los fragmentos proporcionados.,[will y gared],[],[will y gared],0.000000,bad,0
2,gemma3_4b_4bit,3,¿Qué criaturas aparecen más allá del Muro y matan a Ser Waymar Royce?,Los Otros.,Otros.,[los otros],[],[los otros],0.000000,bad,0
3,gemma3_4b_4bit,4,¿Qué le ocurre a Will al final del prólogo?,Es atacado por el cadáver reanimado de Ser Waymar Royce.,El Fragmento 1 describe que el cadáver de Royce comienza a asfixiarle hasta la muerte. visualización,"[waymar royce, el cadaver reanimado, de ser waymar, atacado]",[],"[waymar royce, el cadaver reanimado, de ser waymar, atacado]",0.000000,bad,0
4,gemma3_4b_4bit,5,¿Quién ejecuta a Gared por desertar de la Guardia de la Noche?,Eddard Stark.,Eddard Stark.,[eddard stark],[eddard stark],[],1.000000,good,1
5,gemma3_4b_4bit,6,¿Qué animales muertos encuentran los Stark cerca de los cachorros de huargo?,Una loba huargo y un venado.,Un lobo muerto y un carnicero.,"[loba huargo, y un venado]",[],"[loba huargo, y un venado]",0.000000,bad,0
6,gemma3_4b_4bit,7,¿Cuántos cachorros de huargo encuentran finalmente los Stark?,Seis: cinco primero y después el cachorro albino de Jon.,No lo sé con los fragmentos proporcionados.,"[seis, seis cinco primero, jon, despues el cachorro, el cachorro albino, primero y despues, albino de jon]",[],"[seis, seis cinco primero, jon, despues el cachorro, el cachorro albino, primero y despues, albino de jon]",0.000000,bad,0
7,gemma3_4b_4bit,8,¿Qué noticia recibe Ned Stark sobre Jon Arryn?,Que Jon Arryn ha muerto.,La carta indica que Jon Arryn fue asesinado por los Lannister.cppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcppcpp,"[jon arryn, arryn ha muerto]",[jon arryn],[arryn ha muerto],0.500000,good,1
8,gemma3_4b_4bit,9,¿Qué cargo le ofrece Robert Baratheon a Ned Stark?,Mano del Rey.,Mano del Rey.ัศน์,[mano del rey],[mano del rey],[],1.000000,good,1
9,gemma3_4b_4bit,10,¿Dónde hablan Robert y Ned sobre Lyanna Stark?,En las criptas de Invernalia.,No lo sé con los fragmentos proporcionados.,"[invernalia, criptas de invernalia, las criptas]",[],"[invernalia, criptas de invernalia, las criptas]",0.000000,bad,0


In [71]:
# =========================
# NGRAM SUMMARY BY MODEL
# =========================

ngram_summary = (
    df_eval
    .groupby(["model_alias", "model_id", "quant"], dropna=False)
    .agg(
        n=("idx", "count"),
        ngram_good=("ngram_eval_decision", lambda x: (x == "good").sum()),
        ngram_bad=("ngram_eval_decision", lambda x: (x == "bad").sum()),
        ngram_unknown=("ngram_eval_decision", lambda x: (x == "unknown").sum()),
        ngram_accuracy=("ngram_eval_correct", "mean"),
        mean_ngram_overlap=("ngram_overlap_ratio", "mean"),
        median_ngram_overlap=("ngram_overlap_ratio", "median"),
    )
    .reset_index()
)

ngram_summary["ngram_good_rate"] = ngram_summary["ngram_good"] / ngram_summary["n"]
ngram_summary["ngram_bad_rate"] = ngram_summary["ngram_bad"] / ngram_summary["n"]
ngram_summary["ngram_unknown_rate"] = ngram_summary["ngram_unknown"] / ngram_summary["n"]

ngram_summary = ngram_summary.sort_values(
    by=["ngram_accuracy", "mean_ngram_overlap"],
    ascending=[False, False]
)

display(ngram_summary)

,model_alias,model_id,quant,n,ngram_good,ngram_bad,ngram_unknown,ngram_accuracy,mean_ngram_overlap,median_ngram_overlap,ngram_good_rate,ngram_bad_rate,ngram_unknown_rate
2,gemma4_31b_4bit,google/gemma-4-31b-it,4bit,105,53,52,0,0.504762,0.519331,0.500000,0.504762,0.495238,0.000000
0,gemma3_4b_4bit,google/gemma-3-4b-it,4bit,105,37,68,0,0.352381,0.357902,0.000000,0.352381,0.647619,0.000000
4,gemma4_e4b_bf16,google/gemma-4-E4B-it,bf16,105,35,70,0,0.333333,0.346111,0.000000,0.333333,0.666667,0.000000
1,gemma3_4b_bf16,google/gemma-3-4b-it,bf16,105,33,72,0,0.314286,0.327517,0.000000,0.314286,0.685714,0.000000
3,gemma4_e4b_4bit,google/gemma-4-E4B-it,4bit,105,29,76,0,0.276190,0.276587,0.000000,0.276190,0.723810,0.000000


A simple vista, se ve que este metodo de evaluación arroja resultado mucho más negativos que el anterior de la distancia coseno. Vemos que en este caso la precisión más alta es del 50.4%, cuando antes la más baja era del 65%. Para no hacer un análisis tan extenso como el anterior, vamos a ver si, algunas de las respuestas antes catalogadas como negativas, se han catalogado como positivas con este método:

In [72]:
# =========================
# COSINE BAD vs NGRAM EVALUATION
# =========================

# Filas que cosine/cross ha marcado como incorrectas
# Ajusta el nombre si en tu código la columna final se llama distinto.
bad_cosine_df = df_eval[
    df_eval["generation_eval_decision"] == "bad"
].copy()

print("Casos incorrectos según cosine/cross:", len(bad_cosine_df))

# Comparación con evaluación por n-gramas
bad_cosine_vs_ngrams = bad_cosine_df[
    [
        "model_alias",
        "idx",
        "question",
        "gold_answer",
        "generated_answer",
        "answer_cosine_similarity",
        "generation_eval_decision",
        "gold_keyphrases",
        "matched_keyphrases",
        "missing_keyphrases",
        "ngram_overlap_ratio",
        "ngram_eval_decision",
        "ngram_eval_correct"
    ]
].copy()


Casos incorrectos según cosine/cross: 141


In [73]:
# =========================
# COSINE BAD BUT NGRAM GOOD
# =========================

cosine_bad_ngram_good = df_eval[
    (df_eval["generation_eval_decision"] == "bad") &
    (df_eval["ngram_eval_decision"] == "good")
].copy()

print("Cosine/cross = bad, pero n-gramas = good:", len(cosine_bad_ngram_good))

display(cosine_bad_ngram_good[
    [
        "model_alias",
        "idx",
        "question",
        "gold_answer",
        "generated_answer",
        "answer_cosine_similarity",
        "generation_eval_decision",
        "gold_keyphrases",
        "matched_keyphrases",
        "missing_keyphrases",
        "ngram_overlap_ratio",
        "ngram_eval_decision"
    ]
])

Cosine/cross = bad, pero n-gramas = good: 2


,model_alias,idx,question,gold_answer,generated_answer,answer_cosine_similarity,generation_eval_decision,gold_keyphrases,matched_keyphrases,missing_keyphrases,ngram_overlap_ratio,ngram_eval_decision
310,gemma4_31b_4bit,101,¿Como muere sansa?,No lo sé con los fragmentos proporcionados.,"thought\nNo lo sé con los fragmentos proporcionados.\n---\n**Explicación:** En los textos suministrados, Sansa no muere; se describe su situación como prisionera en el Torreón de Maegor y sus interacciones con Joffrey, pero no se menciona ningún fallecimiento del personaje.\n---\n**Nota:** Según las instrucciones, si la respuesta no se encuentra en los fragment",0.323517,bad,[los fragmentos proporcionados],[los fragmentos proporcionados],[],1.000000,good
389,gemma4_e4b_4bit,75,¿Quiénes capturan a Bran durante su primera salida a caballo?,"Un grupo de salvajes, entre ellos Osha.","fragmento 1 | Fuente=libro | Capítulo=BRAN (5) | POV=BRAN | Score=0.461416\nChapter: BRAN (5)\nChapter order: 37\nPOV: BRAN\nCharacters: Bran, Robb, Jon, Theon, Bailarina, Benjen Stark, Lord Eddard, Stiv, Osha, Hali\nHouses: Stark, Tully",0.388496,bad,"[osha, grupo de salvajes]",[osha],[grupo de salvajes],0.500000,good


Como vemos, solo dos tienen incoherencias. La primera si que estaba mal clasificada, solo que el RAG no ha contestado como se esperaba. En cambio, en la segunda, la respuesta no es correcta, pues está respondiendo algo que tiene sentido, y como aparece la palabra "osha", ya lo da por bueno.

Es decir, podemos concluir que este método no es especialmetne útil.



El siguiente método que utilizaremos para evaluar el RAG será medir el "faithfulness". Es decir, vamos a ver si la respuesta realmente se apoya o no en el contexto recuperado. De esta forma, nos aseguraremos de que el modelo está respondiendo en realidad con el contexto proporcionado, y no gracias a conocimientos que haya adquirido a lo largo de su entrenamiento. 

Para medir el faithfulness, vamos a utilizar dos métodos:


1. Faithfulness por cosine: Calculamos la distancia coseno entre la respuesta y todos los chunks recuperados. Nos quedamos con la mayor distancia, y si esta sobrepasa un umbral, consideramos que la respuesta está respaldada por el texto.

2. Faithfulness por cross-encoder: Calculamos el cross-encoder de todos los pares (chunk, respuesta) y nos quedamos con el que tenga mayor valor.

In [74]:
# =========================
# CONTEXT EXTRACTION
# =========================

def chunk_to_text(chunk):
    if not isinstance(chunk, dict):
        return ""

    text = chunk.get("text", "")
    source = chunk.get("source", "")
    rank_global = chunk.get("rank_global", chunk.get("rank", ""))

    return f"[Chunk {rank_global} | source={source}]\n{text}"


def build_context_from_chunks(chunks, max_chars=None):
    if not isinstance(chunks, list):
        return ""

    parts = []
    total = 0

    for chunk in chunks:
        text = chunk_to_text(chunk)

        if not text:
            continue

        if max_chars is not None and total + len(text) > max_chars:
            remaining = max_chars - total

            if remaining <= 0:
                break

            parts.append(text[:remaining])
            break

        parts.append(text)
        total += len(text)

    return "\n\n---\n\n".join(parts)


def get_chunk_texts(chunks):
    if not isinstance(chunks, list):
        return []

    texts = []

    for chunk in chunks:
        if isinstance(chunk, dict):
            text = chunk.get("text", "")
            if text:
                texts.append(str(text))

    return texts


df_eval["retrieved_context"] = df_eval["retrieved_chunks"].apply(
    lambda x: build_context_from_chunks(x, max_chars=None)
)

df_eval["retrieved_context_lettuce"] = df_eval["retrieved_chunks"].apply(
    lambda x: build_context_from_chunks(x, max_chars=LETTUCE_MAX_CONTEXT_CHARS)
)


### Faithfulles por coseno.

Vamos a calcular la distancia coseno entre el la respuesta generada por el RAG y todos los chunks recuperados. Nos quedamos con la mayor de las distancias, y si llega a un umbral (0.35), lo daremos por bueno.

Si bien el umbral puede parecer bajo, la experiencia nos dice que no. Además, los chunks son muy largos (6000 caracteres), por lo que gran parte del chunk no será necesario para responder, haciendo que la distancia pueda disminuir.

In [75]:
# =========================
# FAITHFULNESS BY COSINE
# =========================

faith_embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=device
)

generated_answers = df_eval["generated_answer"].fillna("").astype(str).tolist()

generated_embs = faith_embedding_model.encode(
    generated_answers,
    batch_size=EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

faith_cosine_max_scores = []
faith_cosine_best_chunk_indices = []

for row_idx, row in tqdm(df_eval.iterrows(), total=len(df_eval)):
    chunk_texts = get_chunk_texts(row["retrieved_chunks"])

    if len(chunk_texts) == 0:
        faith_cosine_max_scores.append(np.nan)
        faith_cosine_best_chunk_indices.append(None)
        continue

    chunk_embs = faith_embedding_model.encode(
        chunk_texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    sims = np.dot(chunk_embs, generated_embs[row_idx])
    best_idx = int(np.argmax(sims))
    best_score = float(sims[best_idx])

    faith_cosine_max_scores.append(best_score)
    faith_cosine_best_chunk_indices.append(best_idx)

df_eval["faith_cosine_max_score"] = faith_cosine_max_scores
df_eval["faith_cosine_best_chunk_idx"] = faith_cosine_best_chunk_indices

df_eval["faith_cosine_decision"] = np.where(
    df_eval["faith_cosine_max_score"] >= FAITH_COSINE_THRESHOLD,
    "good",
    "bad"
)

df_eval.loc[df_eval["faith_cosine_max_score"].isna(), "faith_cosine_decision"] = "unknown"
df_eval["faith_cosine_correct"] = df_eval["faith_cosine_decision"].map({"good": 1, "bad": 0})



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

  0%|          | 0/525 [00:00<?, ?it/s]

In [76]:
display(df_eval[
    [
        "model_alias",
        "idx",
        "question",
        "generated_answer",
        "faith_cosine_max_score",
        "faith_cosine_best_chunk_idx",
        "faith_cosine_decision"
    ]
].head(6))

,model_alias,idx,question,generated_answer,faith_cosine_max_score,faith_cosine_best_chunk_idx,faith_cosine_decision
0,gemma3_4b_4bit,1,¿Quién lidera la partida de exploradores de la Guardia de la Noche en el prólogo?,Jon.,0.537040,0,good
1,gemma3_4b_4bit,2,¿Cómo se llaman los otros dos exploradores que acompañan a Ser Waymar Royce?,No lo sé con los fragmentos proporcionados.,0.239950,4,bad
2,gemma3_4b_4bit,3,¿Qué criaturas aparecen más allá del Muro y matan a Ser Waymar Royce?,Otros.,0.293837,4,bad
3,gemma3_4b_4bit,4,¿Qué le ocurre a Will al final del prólogo?,El Fragmento 1 describe que el cadáver de Royce comienza a asfixiarle hasta la muerte. visualización,0.625581,0,good
4,gemma3_4b_4bit,5,¿Quién ejecuta a Gared por desertar de la Guardia de la Noche?,Eddard Stark.,0.648157,4,good
5,gemma3_4b_4bit,6,¿Qué animales muertos encuentran los Stark cerca de los cachorros de huargo?,Un lobo muerto y un carnicero.,0.429384,4,good


Como vemos, este método no es perfecto. La primera respuesta está mal, pero la da por correcta, y la tercera respuesta está bien, pero la da por incorrecta. No obstante, tampoco nos vamos a preocupar mucho, pues tenemos más métodos con los que comparar.

### Faithfulness por Cross-Encoder

En este método, hemos decidido poner de umbral 0. Como antes, calcularemos el Cross-Encoder de todos los chunks con la respuesta y nos quedaremos con el mayor. Si supera 0, diremos que es correcta la respuesta.

In [77]:
# =========================
# FAITHFULNESS BY CROSS-ENCODER
# =========================

faith_cross_encoder = CrossEncoder(
    FAITHFULNESS_CROSS_ENCODER_MODEL_NAME,
    device=device
)

faith_ce_max_scores = []
faith_ce_best_chunk_indices = []

for _, row in tqdm(df_eval.iterrows(), total=len(df_eval)):
    generated = str(row["generated_answer"])
    chunk_texts = get_chunk_texts(row["retrieved_chunks"])

    if len(chunk_texts) == 0:
        faith_ce_max_scores.append(np.nan)
        faith_ce_best_chunk_indices.append(None)
        continue

    pairs = [(generated, chunk_text) for chunk_text in chunk_texts]

    scores = faith_cross_encoder.predict(
        pairs,
        batch_size=CROSS_ENCODER_BATCH_SIZE,
        show_progress_bar=False
    )

    best_idx = int(np.argmax(scores))
    best_score = float(scores[best_idx])

    faith_ce_max_scores.append(best_score)
    faith_ce_best_chunk_indices.append(best_idx)

df_eval["faith_cross_encoder_max_score"] = faith_ce_max_scores
df_eval["faith_cross_encoder_best_chunk_idx"] = faith_ce_best_chunk_indices

df_eval["faith_cross_encoder_decision"] = np.where(
    df_eval["faith_cross_encoder_max_score"] >= FAITH_CROSS_ENCODER_THRESHOLD,
    "good",
    "bad"
)

df_eval.loc[df_eval["faith_cross_encoder_max_score"].isna(), "faith_cross_encoder_decision"] = "unknown"
df_eval["faith_cross_encoder_correct"] = df_eval["faith_cross_encoder_decision"].map({"good": 1, "bad": 0})

display(df_eval[
    [
        "model_alias",
        "idx",
        "question",
        "generated_answer",
        "faith_cross_encoder_max_score",
        "faith_cross_encoder_best_chunk_idx",
        "faith_cross_encoder_decision"
    ]
].head(6))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/525 [00:00<?, ?it/s]

,model_alias,idx,question,generated_answer,faith_cross_encoder_max_score,faith_cross_encoder_best_chunk_idx,faith_cross_encoder_decision
0,gemma3_4b_4bit,1,¿Quién lidera la partida de exploradores de la Guardia de la Noche en el prólogo?,Jon.,3.738573,0,good
1,gemma3_4b_4bit,2,¿Cómo se llaman los otros dos exploradores que acompañan a Ser Waymar Royce?,No lo sé con los fragmentos proporcionados.,-5.140645,2,bad
2,gemma3_4b_4bit,3,¿Qué criaturas aparecen más allá del Muro y matan a Ser Waymar Royce?,Otros.,-1.358629,2,bad
3,gemma3_4b_4bit,4,¿Qué le ocurre a Will al final del prólogo?,El Fragmento 1 describe que el cadáver de Royce comienza a asfixiarle hasta la muerte. visualización,0.656785,0,good
4,gemma3_4b_4bit,5,¿Quién ejecuta a Gared por desertar de la Guardia de la Noche?,Eddard Stark.,4.455112,3,good
5,gemma3_4b_4bit,6,¿Qué animales muertos encuentran los Stark cerca de los cachorros de huargo?,Un lobo muerto y un carnicero.,1.612947,4,good


Al igual que antes, hemos conseguido exactamente lo mismo resultados. Nuevamente, la primera respuesta está mal, y la tercera está bien, pero nuestros métodos dicen lo contrario. Más adelante, veremos qué % de coincidencia tienen estos dos métodos con los métodos de evaluación anteriores (distancia coseno, ngramas...), y valoraremos.

### Comparación entre Cosine y Cross

In [78]:
# =========================
# ASEGURAR COLUMNAS
# =========================

required_cols = [
    "model_alias",
    "faith_cosine_decision",
    "faith_cross_encoder_decision",
    "faith_cosine_max_score",
    "faith_cross_encoder_max_score"
]

for col in required_cols:
    if col not in df_eval.columns:
        raise ValueError(f"Falta columna: {col}")

# =========================
# AGREEMENT ENTRE MÉTODOS
# =========================

df_eval["faith_agree"] = (
    df_eval["faith_cosine_decision"] == df_eval["faith_cross_encoder_decision"]
)

df_eval["faith_both_good"] = (
    (df_eval["faith_cosine_decision"] == "good") &
    (df_eval["faith_cross_encoder_decision"] == "good")
)

df_eval["faith_both_bad"] = (
    (df_eval["faith_cosine_decision"] == "bad") &
    (df_eval["faith_cross_encoder_decision"] == "bad")
)

df_eval["faith_conflict"] = (
    df_eval["faith_cosine_decision"] != df_eval["faith_cross_encoder_decision"]
)

# =========================
# RESUMEN POR MODELO
# =========================

summary_faith = df_eval.groupby("model_alias").agg(

    n=("model_alias", "count"),

    # tasas individuales
    cosine_good_rate=("faith_cosine_decision", lambda x: (x == "good").mean()),
    cross_good_rate=("faith_cross_encoder_decision", lambda x: (x == "good").mean()),

    # acuerdo
    agreement_rate=("faith_agree", "mean"),
    both_good_rate=("faith_both_good", "mean"),
    both_bad_rate=("faith_both_bad", "mean"),
    conflict_rate=("faith_conflict", "mean"),

    # scores (opcional pero útil)
    mean_cosine_score=("faith_cosine_max_score", "mean"),
    mean_cross_score=("faith_cross_encoder_max_score", "mean"),

).reset_index()

display(summary_faith)

,model_alias,n,cosine_good_rate,cross_good_rate,agreement_rate,both_good_rate,both_bad_rate,conflict_rate,mean_cosine_score,mean_cross_score
0,gemma3_4b_4bit,105,0.761905,0.628571,0.828571,0.609524,0.219048,0.171429,0.499267,0.961800
1,gemma3_4b_bf16,105,0.790476,0.628571,0.723810,0.571429,0.152381,0.276190,0.498866,1.116806
2,gemma4_31b_4bit,105,0.980952,0.676190,0.676190,0.666667,0.009524,0.323810,0.737780,1.644735
3,gemma4_e4b_4bit,105,0.952381,0.600000,0.647619,0.600000,0.047619,0.352381,0.695378,0.985725
4,gemma4_e4b_bf16,105,0.761905,0.504762,0.742857,0.504762,0.238095,0.257143,0.564174,0.324103


Como antes, vamos a hacer el análisis por modelos:

- Gemma-4 31B 4bit:
    - Según el primer método (distancia coseno), el 98% de las respuestas son correctas. En el metodo anterior, en el que comparabamos la respuesta golden con la respeusta del RAG, este porcentaje era del 88%. En cambio, con el método cross, el porcentaje de aciertos baja al 67.6%, muy cercano al límite inferior del anterior método. Está claro que hay muchas discrepancias entre metodos.
    - Hay una discrepancia del 32% entre los dos métodos.
- Gemma-4 E4B 4-bit:
    - Según el primer método, el % de aciertos es del 95% en esta ocasión, mientras que con el segundo es del 60%. Nuevamente, tenemos una fuerte discrepancia (el 35%). Además, comparando con métodos anteriores, este modelo tenía un % de aciertos del 70% aproximadamente.
- Gemma-3 4B bf16:
    - Esta vez tenemos un porcentaje de aciertos algo más realista (aunque este modelo es peor, así que en realidad sigue siendo alto comparando con el método anterior). Los dos porcentajes de aciertos son del casi 80% y del casi 63%.

- Gemma-3 4B 4-bit:
    - Como antes, los porcentajes son parecidos: 76% y 63%. Más realista, y esta vez con menos discrepancia (el 14%).

- Gemma-4 E4B bf16:
    - Curiosamente, como antes, el que debería ser el segundo mejor modelo, tiene los peores resultados. Tenemos un 76% de aciertos con el método coseno pero solo un 50% con el método cross. No obstante, viendo resultados previos, este es el modelo que parece tener resultados más "realistas".

Para acabar con este análisis, veamos los resultados de comparar cada método de evaluación por modelo:

In [79]:

# =========================
# FLAGS DE CONSISTENCIA
# =========================

df_eval["all_agree_good"] = (
    (df_eval["generation_eval_decision"] == "good") &
    (df_eval["ngram_eval_decision"] == "good") &
    (df_eval["faith_cosine_decision"] == "good") &
    (df_eval["faith_cross_encoder_decision"] == "good")
)

df_eval["all_agree_bad"] = (
    (df_eval["generation_eval_decision"] == "bad") &
    (df_eval["ngram_eval_decision"] == "bad") &
    (df_eval["faith_cosine_decision"] == "bad") &
    (df_eval["faith_cross_encoder_decision"] == "bad")
)

# =========================
# CONFLICTOS CONTRA COSINE/CROSS DE GENERACIÓN
# =========================

df_eval["conflict_cosine_ngram"] = (
    (df_eval["generation_eval_decision"] == "good") &
    (df_eval["ngram_eval_decision"] == "bad")
)

df_eval["conflict_cosine_faith_cosine"] = (
    (df_eval["generation_eval_decision"] == "good") &
    (df_eval["faith_cosine_decision"] == "bad")
)

df_eval["conflict_cosine_faith_cross"] = (
    (df_eval["generation_eval_decision"] == "good") &
    (df_eval["faith_cross_encoder_decision"] == "bad")
)

df_eval["conflict_ngram_cosine"] = (
    (df_eval["generation_eval_decision"] == "bad") &
    (df_eval["ngram_eval_decision"] == "good")
)

df_eval["conflict_faith_cosine_cosine"] = (
    (df_eval["generation_eval_decision"] == "bad") &
    (df_eval["faith_cosine_decision"] == "good")
)

df_eval["conflict_faith_cross_cosine"] = (
    (df_eval["generation_eval_decision"] == "bad") &
    (df_eval["faith_cross_encoder_decision"] == "good")
)

# =========================
# AGREEMENTS CONTRA COSINE/CROSS DE GENERACIÓN
# =========================

df_eval["agree_cosine_ngram"] = (
    df_eval["generation_eval_decision"] == df_eval["ngram_eval_decision"]
)

df_eval["agree_cosine_faith_cosine"] = (
    df_eval["generation_eval_decision"] == df_eval["faith_cosine_decision"]
)

df_eval["agree_cosine_faith_cross"] = (
    df_eval["generation_eval_decision"] == df_eval["faith_cross_encoder_decision"]
)

# =========================
# RESUMEN POR MODELO
# =========================

summary_by_model = df_eval.groupby("model_alias").agg(

    n=("model_alias", "count"),

    # --- tasas individuales ---
    cosine_good_rate=("generation_eval_decision", lambda x: (x == "good").mean()),
    ngram_good_rate=("ngram_eval_decision", lambda x: (x == "good").mean()),
    faith_cosine_good_rate=("faith_cosine_decision", lambda x: (x == "good").mean()),
    faith_cross_good_rate=("faith_cross_encoder_decision", lambda x: (x == "good").mean()),

    # --- agreement contra cosine/cross de generación ---
    agree_cosine_ngram_rate=("agree_cosine_ngram", "mean"),
    agree_cosine_faith_cosine_rate=("agree_cosine_faith_cosine", "mean"),
    agree_cosine_faith_cross_rate=("agree_cosine_faith_cross", "mean"),

    # --- consenso global ---
    all_agree_good_rate=("all_agree_good", "mean"),
    all_agree_bad_rate=("all_agree_bad", "mean"),

    # --- conflictos: cosine good pero otro método bad ---
    conflict_cosine_ngram_rate=("conflict_cosine_ngram", "mean"),
    conflict_cosine_faith_cosine_rate=("conflict_cosine_faith_cosine", "mean"),
    conflict_cosine_faith_cross_rate=("conflict_cosine_faith_cross", "mean"),

    # --- conflictos inversos: cosine bad pero otro método good ---
    conflict_ngram_cosine_rate=("conflict_ngram_cosine", "mean"),
    conflict_faith_cosine_cosine_rate=("conflict_faith_cosine_cosine", "mean"),
    conflict_faith_cross_cosine_rate=("conflict_faith_cross_cosine", "mean"),

).reset_index()

display(summary_by_model)

,model_alias,n,cosine_good_rate,ngram_good_rate,faith_cosine_good_rate,faith_cross_good_rate,agree_cosine_ngram_rate,agree_cosine_faith_cosine_rate,agree_cosine_faith_cross_rate,all_agree_good_rate,all_agree_bad_rate,conflict_cosine_ngram_rate,conflict_cosine_faith_cosine_rate,conflict_cosine_faith_cross_rate,conflict_ngram_cosine_rate,conflict_faith_cosine_cosine_rate,conflict_faith_cross_cosine_rate
0,gemma3_4b_4bit,105,0.695238,0.352381,0.761905,0.628571,0.657143,0.780952,0.704762,0.247619,0.161905,0.342857,0.076190,0.180952,0.000000,0.142857,0.114286
1,gemma3_4b_bf16,105,0.733333,0.314286,0.790476,0.628571,0.580952,0.752381,0.628571,0.190476,0.095238,0.419048,0.095238,0.238095,0.000000,0.152381,0.133333
2,gemma4_31b_4bit,105,0.885714,0.504762,0.980952,0.676190,0.600000,0.866667,0.638095,0.342857,0.000000,0.390476,0.019048,0.285714,0.009524,0.114286,0.076190
3,gemma4_e4b_4bit,105,0.685714,0.276190,0.952381,0.600000,0.571429,0.695238,0.476190,0.161905,0.028571,0.419048,0.019048,0.304762,0.009524,0.285714,0.219048
4,gemma4_e4b_bf16,105,0.657143,0.333333,0.761905,0.504762,0.676190,0.800000,0.580952,0.161905,0.190476,0.323810,0.047619,0.285714,0.000000,0.152381,0.133333


En esta tabla de aquí podemos ver el resumen de todos los métodos de evaluación aplicados a las respuestas que nos ha dado cada modelo.

Las primeras cuatro columnas son las ya comentadas anteriormente, mientras que las demás son comparaciones entre los diferentes métodos de evaluación.

Al igual que antes, si cogemos los resutlados de cada método por separados, el mejor modelo es el Gemma-4 31B 4-bit, pues gana en todas las métricas a la vez. Es el modelo más grande que hemos utilizado y el más reciente de Google.

Por experiencia de pruebas que hemos hecho con anterioridad, consideramos que la métrica más importante y más fiel a la realidad es la de la distancia coseno entre la respuesta gold y la respuesta RAG. Por eso mismo, el análisis se va a centrar en la comparación de esta métrica con el resto.

- Gemma-4 31B 4-bit:

    - El ratio de concordancia entre el método coseno y el método de n-gramas es del 60%. No está nada mal, teniendo en cuenta que esta era, con diferencia el peor de los metodos debido a su simpleza.
    - Si comparamos el ratio de correctos entre la distancia coseno de las respuestas y la distancia coseno de los chunks, tenemos que coinciden el 87% de las veces. Este valor es muy alto, pero también hay que tener en cuenta que el método de distancia con chunks tiene un porcentaje de aciertos exageradamente alto.
    - Por otro lado, si lo comparamos con el cross-encoder entre chunks, vemos que el ratio de concordancia baja hasta el 64%. Mucho más bajo. Sigue siendo superior al 50%, pero está peligrosamente cerca de ser un mal porcentaje.
    - Podemos concluir que, si bien entre metodos la concordancia no es especialmente alta, las respuestas del RAG son en general buenas. Además, la experiencia de haber usado extensamente este modelo así lo corrobora.

- Gemma-3 4B 4-bit:

    - El ratio de concordancia entre el método coseno y el método de n-gramas es del **65.7%**, ligeramente superior al del modelo grande. Esto puede parecer contraintuitivo, pero en realidad refleja que ambos métodos están capturando errores similares, aunque el modelo en sí sea peor.
    - Si comparamos cosine vs faithfulness por cosine (respuesta vs chunks), la concordancia sube hasta el **78.1%**, un valor bastante alto. Esto indica que cuando el modelo acierta o falla, normalmente está alineado con el contexto recuperado.
    - Sin embargo, al comparar con el cross-encoder de chunks, la concordancia baja al **70.5%**, lo que ya empieza a mostrar inconsistencias entre métricas más semánticas.
    - Un dato importante: el conflicto cosine–ngram es del **34.3%**, bastante alto. Esto refuerza la idea de que el método de n-gramas es débil y no fiable como métrica principal.
    - En general, el modelo funciona razonablemente bien, pero muestra más inconsistencias que el 31B, especialmente cuando usamos evaluaciones más exigentes.


- Gemma-3 4B BF16:

    - La concordancia cosine–ngram baja al **58.1%**, peor que su versión 4-bit. Esto es interesante porque sugiere que el cambio de cuantización no mejora la consistencia entre métricas.
    - En cosine vs faithfulness (cosine chunks), obtenemos un **75.2%**, ligeramente inferior al modelo anterior pero todavía alto.
    - Con cross-encoder, la concordancia cae a **62.9%**, bastante baja. Aquí ya vemos una divergencia clara entre métricas más simples y más sofisticadas.
    - El conflicto cosine–ngram es del **41.9%**, uno de los más altos de todos los modelos. Esto refuerza aún más que el método de n-gramas introduce bastante ruido.
    - En conjunto, este modelo es algo peor y más inconsistente que su versión 4-bit, lo cual es curioso y probablemente tenga que ver con cómo está optimizado el modelo en esa configuración.


- Gemma-4 E4B 4-bit:

    - La concordancia cosine–ngram es del **57.1%**, bastante baja. De nuevo, el método de n-gramas no parece alinearse bien con cosine.
    - En cosine vs faithfulness (cosine chunks), tenemos un **69.5%**, claramente inferior al resto de modelos. Esto sugiere que muchas respuestas “correctas” según cosine no están bien soportadas por el contexto.
    - Con cross-encoder, la concordancia cae aún más hasta **47.6%**, por debajo del 50%. Esto es una señal clara de inconsistencia fuerte entre métricas.
    - El conflicto cosine–faith_cross es del **30.5%**, bastante alto, lo que indica que el modelo genera respuestas que parecen correctas superficialmente pero no están bien fundamentadas.
    - En resumen, este modelo es el más inconsistente de todos: aunque su accuracy no es terrible, la falta de alineación entre métricas indica problemas claros de grounding.


- Gemma-4 E4B BF16:

    - La concordancia cosine–ngram sube a **67.6%**, siendo de las más altas. De nuevo, esto no implica que sea mejor modelo, sino que ambos métodos coinciden más (aunque sea en errores).
    - En cosine vs faithfulness (cosine chunks), obtenemos un **80%**, uno de los valores más altos. Esto indica buena alineación entre respuesta y contexto.
    - Con cross-encoder, la concordancia baja a **58.1%**, lo que muestra que al exigir más precisión semántica, el modelo falla más.
    - El conflicto cosine–faith_cross es del **28.6%**, elevado, lo que apunta a problemas de fidelidad semántica real.
    - En conjunto, este modelo está en un punto intermedio: mejor que el E4B 4-bit en consistencia, pero lejos del comportamiento del 31B.


- Conclusión global

    - El modelo **Gemma-4 31B 4-bit** no solo gana en métricas individuales, sino que también mantiene una **consistencia razonable entre métodos**, especialmente con cosine como referencia.
    - Los modelos pequeños (**Gemma-3 4B**) muestran una **consistencia aceptable**, pero con mayor ruido, especialmente en comparación con métodos más semánticos.
    - Los modelos E4B presentan el mayor problema: **alta variabilidad entre métricas**, lo que indica que generan respuestas plausibles pero menos fundamentadas.
    - El método de **n-gramas** confirma lo esperado: tiene baja alineación con cosine y altos conflictos, por lo que debe considerarse **auxiliar, no principal**.
    - El **cross-encoder aplicado a faithfulness** es claramente más exigente: reduce las tasas de concordancia y expone debilidades que cosine no detecta.


# Evaluación de la recuperación de Chunks

Hasta ahora hemos evaluado la *alidad de las respuestas generadas, pero falta analizar una parte importante del sistema: el retrieval o la recuperación del contexto.

En esta fase vamos a medir qué tan bien el sistema recupera los fragmentos relevantes para responder a cada pregunta. Para ello utilizaremos métricas estándar en Information Retrieval:

### Métricas que vamos a implementar

- Recall@k
  - Mide si el fragmento correcto aparece dentro de los *k* chunks recuperados.
  - Es una métrica de cobertura:  
    > ¿Está la información correcta entre los resultados?

- MRR (Mean Reciprocal Rank)
  - Evalúa en qué posición aparece el primer chunk relevante.
  - Penaliza que el chunk correcto esté muy abajo en el ranking.

- Context Precision
  - Mide qué proporción de los chunks recuperados son realmente relevantes.
  - Es una métrica de calidad del conjunto recuperado.

### Objetivo

Con estas métricas queremos responder a preguntas como:

- ¿El modelo está recuperando el contexto correcto?
- ¿Lo está recuperando en buena posición?
- ¿El contexto recuperado está limpio o lleno de ruido?

Esto nos permitirá separar claramente dos problemas:

- Mal retrieval → el modelo no tiene información correcta  
- Mala generación → el modelo falla aunque el contexto sea bueno  

Esta distinción es clave para mejorar el sistema RAG de forma dirigida.

### Helpers para extraer el capítulo de cada chunk:

In [80]:
import ast
import re
import pandas as pd

def parse_gold_chapters_value(x):
    """
    Convierte cualquier formato raro de gold_chapters a lista limpia.
    
    Soporta:
    - ["Catelyn VII", "Tyrion V"]
    - "[Catelyn VII, Tyrion V]"
    - "['Catelyn VII', 'Tyrion V']"
    - "Catelyn VII"
    """
    if isinstance(x, list):
        out = []
        for item in x:
            out.extend(parse_gold_chapters_value(item))
        return out

    if pd.isna(x):
        return []

    s = str(x).strip()

    # Intento literal_eval para strings tipo "['A', 'B']"
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return parse_gold_chapters_value(parsed)
    except Exception:
        pass

    # Limpieza manual
    s = s.strip()
    s = s.strip("[]")
    s = s.replace("'", "").replace('"', "")

    parts = [p.strip() for p in s.split(",") if p.strip()]
    return parts

In [81]:
import unicodedata

def strip_accents(text):
    text = unicodedata.normalize("NFD", str(text))
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn")

def normalize_text(text):
    text = "" if text is None else str(text)
    text = strip_accents(text.lower().strip())
    text = re.sub(r"\s+", " ", text)
    return text

    
ROMAN_TO_INT = {
    "i": 1, "ii": 2, "iii": 3, "iv": 4, "v": 5,
    "vi": 6, "vii": 7, "viii": 8, "ix": 9, "x": 10,
    "xi": 11, "xii": 12, "xiii": 13, "xiv": 14, "xv": 15,
    "I": 1, "II": 2, "III": 3, "IV": 4, "V": 5,
    "VI": 6, "VII": 7, "VIII": 8, "IX": 9, "X": 10,
    "XI": 11, "XII": 12, "XIII": 13, "XIV": 14, "XV": 15
}


def parse_gold_chapter(chapter):
    ch = normalize_text(chapter)

    # Limpieza extra por si vienen restos de listas/string
    ch = ch.replace("[", "").replace("]", "")
    ch = ch.replace("'", "").replace('"', "")
    ch = ch.strip(" ,.;:")

    if ch in ["prologo", "prólogo"]:
        return "prologo"

    parts = ch.split()

    if len(parts) >= 2:
        pov = parts[0].strip()
        num = parts[-1].strip(" ,.;:")

        if num in ROMAN_TO_INT:
            return f"{pov}_{ROMAN_TO_INT[num]}"

        if num.isdigit():
            return f"{pov}_{int(num)}"

    return ch


def parse_chunk_key(chunk):
    if not isinstance(chunk, dict):
        return ""

    metadata = chunk.get("metadata", {})

    if not isinstance(metadata, dict):
        return ""

    # Caso libro: "TYRION (5)"
    title = metadata.get("chapter_title", "")
    if title:
        t = normalize_text(title)

        m = re.search(r"([a-z]+)\s*\((\d+)\)", t)
        if m:
            return f"{m.group(1)}_{int(m.group(2))}"

        if t == "prologo":
            return "prologo"

    # Caso resumen
    chapter = metadata.get("chapter", "")
    if chapter:
        ch = normalize_text(chapter)

        if "prologo" in ch:
            return "prologo"

    return ""


def parse_chunk_order(chunk):
    metadata = chunk.get("metadata", {})

    # caso libro
    if "chapter_order" in metadata:
        try:
            return int(metadata["chapter_order"])
        except:
            pass

    # caso resumen
    chapter = metadata.get("chapter", "")
    if chapter:
        ch = normalize_text(chapter)
        m = re.search(r"capitulo\s*(\d+)", ch)
        if m:
            return int(m.group(1))

    return None


chapter_key_to_order = {}

for chunks in df_eval["retrieved_chunks"]:
    if not isinstance(chunks, list):
        continue

    for chunk in chunks:
        key = parse_chunk_key(chunk)
        order = parse_chunk_order(chunk)

        if key and order is not None:
            if not key.startswith("chapter_order_"):
                chapter_key_to_order[key] = order

In [82]:
def chunk_matches_gold(chunk, gold_chapters):
    gold_chapters = parse_gold_chapters_value(gold_chapters)

    if len(gold_chapters) == 0:
        return False

    chunk_key = parse_chunk_key(chunk)
    chunk_order = parse_chunk_order(chunk)

    gold_keys = [parse_gold_chapter(g) for g in gold_chapters]

    for gk in gold_keys:

        # Match directo:
        # "Tyrion V" -> "tyrion_5"
        # "TYRION (5)" -> "tyrion_5"
        if chunk_key == gk:
            return True

        # Match por chapter_order
        gold_order = chapter_key_to_order.get(gk)

        if gold_order is not None and chunk_order is not None:
            if int(gold_order) == int(chunk_order):
                return True

        # Prólogo
        if gk == "prologo" and chunk_key == "prologo":
            return True

    return False

In [ ]:
# =========================
# MARCAR HARD NEGATIVES
# =========================

if "is_hard_negative" in df_eval.columns:
    df_eval["is_hard_negative_eval"] = df_eval["is_hard_negative"].fillna(False).astype(bool)
else:
    df_eval["is_hard_negative_eval"] = False

df_eval.loc[df_eval["idx"].between(101, 105), "is_hard_negative_eval"] = True


df_eval["global_row_1based"] = range(1, len(df_eval) + 1)

hard_negative_global_ranges = [
    (101, 105),
    (206, 210),
    (311, 315), 
    (417, 420),
    (521, 525),
]

for start, end in hard_negative_global_ranges:
    df_eval.loc[
        df_eval["global_row_1based"].between(start, end),
        "is_hard_negative_eval"
    ] = True

print("Hard negatives detectadas:", df_eval["is_hard_negative_eval"].sum())



Hard negatives detectadas: 25


In [84]:
def evaluate_retrieval_row(row):
    chunks = row.get("retrieved_chunks", [])
    gold_chapters = row.get("gold_chapters", [])

    # Hard negatives: no existe chunk correcto.
    # Las excluimos del retrieval normal usando NaN.
    if row.get("is_hard_negative_eval", False):
        return pd.Series({
            "retrieval_recall_at_1": np.nan,
            "retrieval_recall_at_3": np.nan,
            "retrieval_recall_at_5": np.nan,
            "retrieval_mrr": np.nan,
            "retrieval_context_precision": np.nan,
            "retrieval_first_rank": np.nan,
            "retrieval_relevant_count": 0,
            "retrieval_relevance_list": [],
            "hard_negative_retrieval_expected": True
        })

    if not isinstance(chunks, list) or len(chunks) == 0:
        return pd.Series({
            "retrieval_recall_at_1": 0,
            "retrieval_recall_at_3": 0,
            "retrieval_recall_at_5": 0,
            "retrieval_mrr": 0,
            "retrieval_context_precision": 0,
            "retrieval_first_rank": np.nan,
            "retrieval_relevant_count": 0,
            "retrieval_relevance_list": [],
            "hard_negative_retrieval_expected": False
        })

    relevant = [
        chunk_matches_gold(c, gold_chapters)
        for c in chunks
    ]

    recall_at_1 = int(any(relevant[:1]))
    recall_at_3 = int(any(relevant[:3]))
    recall_at_5 = int(any(relevant[:5]))

    first_rank = np.nan
    for i, r in enumerate(relevant):
        if r:
            first_rank = i + 1
            break

    mrr = 0 if pd.isna(first_rank) else 1 / first_rank
    precision = sum(relevant) / len(relevant)

    return pd.Series({
        "retrieval_recall_at_1": recall_at_1,
        "retrieval_recall_at_3": recall_at_3,
        "retrieval_recall_at_5": recall_at_5,
        "retrieval_mrr": mrr,
        "retrieval_context_precision": precision,
        "retrieval_first_rank": first_rank,
        "retrieval_relevant_count": sum(relevant),
        "retrieval_relevance_list": relevant,
        "hard_negative_retrieval_expected": False
    })

In [105]:
df_analysis = df_eval[~df_eval["is_hard_negative_eval"]].copy()

retrieval_summary = pd.DataFrame([{
    "n": len(df_analysis),

    "recall_at_1": df_analysis["retrieval_recall_at_1"].mean(),
    "recall_at_3": df_analysis["retrieval_recall_at_3"].mean(),
    "recall_at_5": df_analysis["retrieval_recall_at_5"].mean(),

    "mrr": df_analysis["retrieval_mrr"].mean(),

    "context_precision": df_analysis["retrieval_context_precision"].mean(),

    "avg_first_rank": df_analysis["retrieval_first_rank"].mean(),
}])

display(retrieval_summary)

,n,recall_at_1,recall_at_3,recall_at_5,mrr,context_precision,avg_first_rank
0,500,0.726000,0.946000,0.966000,0.828833,0.481600,1.372671


En la celda de arriba, podemos ver los resultados de la primera parte de la evaluación del Retrieval de RAG. La evaluación se ha hecho utilizando los .jsonl creados anteriormente, como en el caso de evaluación de las respuestas del RAG. 

Antes de comentar la evaluación, hemos tenido en cuenta una cosa:

- Como hemos añadido Hard-Negatives, es decir, preguntas imposibles de responder porque no suceden en ninguna parte del libro, se han excluido esas preguntas del cálculo. Esto se hace así porque el modelo de Embedding recuperará siempre 5 chunks, pero la respuesta no estará en esos 5 chunks. Por lo tanto, la respuesta será "No puedo responder". Esa respuesta se dará por buena, porque es lo que buscamos, pero el chunk necesario para responder, al no existir, no aparecerá en el top_K. Eso ensuciará los cálculos, por lo que han de eliminarse del cálculo para que todo salga bien.

Teniendo esto en cuenta, vemos que hemos conseguido unos resultados realmente prometedores:

- En el caso de recall_at_5, vemos que en el 96.6% de la veces, hemos recuperado el chunk necesario para responder entre los 5 primeros.
- El 94.6% de la veces, el chunk necesario estaba en la posición 1, 2 o 3.
- El 72.6% de la veces, el top necesario estaba en la primera posición. Es decir, el modelo de embedding ha sabido elegir entre los 320 chunks que tenemos, y ha puesto el chunk en la primera posición.

Esta métricas son realmente buenas. Podemos concluir que añadir los metadatos ha sido muy útil.

Por otro lado, fijándonos en el mrr y en el context_precision, tenemos también buenos resultados:

- Un MRR de 0.82 nos indica que la mayoría de la vecesm el chunk necesario apareció en posiciones altas, siempre entre el top 1 y el top 2.

- Si bien el context_precision puede parecer bajo (0.48), en realidad está bien teniendo en cuenta lo largos que son nuestros chunks (recordemos que estamos trabajando con chunks de 6000 caracteres). Solo una pequeña parte será necesaria para poder contestar la pregunta.

En resumidas cuentas, el Retrieval está funcionando bien.

Ahora, evaluemos el Retrieval y la generación de respuestas a la vez.

Queremos ver si el modelo está contestando bien porque ya lo sabía de antemano (es decir, durante su entrenamiento leyó Juego de Tronos y ya sabía la respuesta), o si ha respondido realmente basándose solo en la información de los chunks. Para ellos, calcularemos las siguiente métricas:

- 	P(cosine=good | hit): Esta métrica representa la probabilidad de que, estando el chunk necesario para responder en el top_K, el modelo sea capaz de responder.
- 	P(cosine=good | miss): Esta métrica representa la probabilidad de que, NO estando el chunk necesario para responder en el top_K, el modelo sea capaz de responder (es decir, que acierte porque ya lo sabía).
También usamos métricas complementarias para entender los resultados.

In [101]:
import pandas as pd

# =========================
# FLAGS
# =========================

df_eval["retrieval_hit@1"] = df_eval["retrieval_recall_at_1"] == 1
df_eval["retrieval_hit@3"] = df_eval["retrieval_recall_at_3"] == 1
df_eval["retrieval_hit@5"] = df_eval["retrieval_recall_at_5"] == 1

df_eval["cosine_good"] = df_eval["cosine_decision"] == "good"

# =========================
# EXCLUIR HARD NEGATIVES
# =========================

df_analysis = df_eval[~df_eval["is_hard_negative_eval"]].copy()

# =========================
# FUNCIÓN
# =========================

def compute_metrics(group, retrieval_col):
    hit = group[group[retrieval_col]]
    miss = group[~group[retrieval_col]]

    return pd.Series({
        "P(cosine=good | hit)": hit["cosine_good"].mean() if len(hit) > 0 else None,
        "P(cosine=good | miss)": miss["cosine_good"].mean() if len(miss) > 0 else None,
        "agreement": (group[retrieval_col] == group["cosine_good"]).mean(),
        "n": len(group),
        "n_hit": len(hit),
        "n_miss": len(miss),
    })

# =========================
# CÁLCULO POR MODELO
# =========================

results = []

for k in ["retrieval_hit@1", "retrieval_hit@3", "retrieval_hit@5"]:
    tmp = df_analysis.groupby("model_alias").apply(
        lambda g: compute_metrics(g, k)
    ).reset_index()

    tmp["k"] = k
    results.append(tmp)

df_retrieval_vs_cosine = pd.concat(results, ignore_index=True)

In [102]:
df_pivot = df_retrieval_vs_cosine.pivot_table(
    index="model_alias",
    columns="k",
    values=[
        "P(cosine=good | hit)",
        "P(cosine=good | miss)",
        "agreement",
        "n_hit",
        "n_miss"
    ]
)

display(df_pivot)

P(cosine=good | hit)                                  \
k                    retrieval_hit@1 retrieval_hit@3 retrieval_hit@5   
model_alias                                                            
gemma3_4b_4bit              0.722222        0.680851        0.687500   
gemma3_4b_bf16              0.736111        0.723404        0.729167   
gemma4_31b_4bit             0.945205        0.905263        0.907216   
gemma4_e4b_4bit             0.643836        0.631579        0.639175   
gemma4_e4b_bf16             0.684932        0.642105        0.639175   

                P(cosine=good | miss)                                  \
k                     retrieval_hit@1 retrieval_hit@3 retrieval_hit@5   
model_alias                                                             
gemma3_4b_4bit               0.571429        0.666667        0.500000   
gemma3_4b_bf16               0.678571        0.666667        0.500000   
gemma4_31b_4bit              0.740741        0.600000        0.333333   
gemma4_e4b_4bit              0.629630        0.800000        0.666667   
gemma4_e4b_bf16              0.481481        0.400000        0.333333   

                      agreement                                  \
k               retrieval_hit@1 retrieval_hit@3 retrieval_hit@5   
model_alias                                                       
gemma3_4b_4bit         0.640000        0.660000        0.680000   
gemma3_4b_bf16         0.620000        0.700000        0.720000   
gemma4_31b_4bit        0.760000        0.880000        0.900000   
gemma4_e4b_4bit        0.570000        0.610000        0.630000   
gemma4_e4b_bf16        0.640000        0.640000        0.640000   

                          n_hit                                  \
k               retrieval_hit@1 retrieval_hit@3 retrieval_hit@5   
model_alias                                                       
gemma3_4b_4bit        72.000000       94.000000       96.000000   
gemma3_4b_bf16        72.000000       94.000000       96.000000   
gemma4_31b_4bit       73.000000       95.000000       97.000000   
gemma4_e4b_4bit       73.000000       95.000000       97.000000   
gemma4_e4b_bf16       73.000000       95.000000       97.000000   

                         n_miss                                  
k               retrieval_hit@1 retrieval_hit@3 retrieval_hit@5  
model_alias                                                      
gemma3_4b_4bit        28.000000        6.000000        4.000000  
gemma3_4b_bf16        28.000000        6.000000        4.000000  
gemma4_31b_4bit       27.000000        5.000000        3.000000  
gemma4_e4b_4bit       27.000000        5.000000        3.000000  
gemma4_e4b_bf16       27.000000        5.000000        3.000000

Como vemos, a simple vista, los resultados no parecen alentadores, pero todo tiene su explicación. Veamos los resultados por modelo.

- 	Gemma-4 31B 4bit:

    - Como con los resultados anteriores, el modelo Gemma 4 de 31B es el mejor de todos. Aunque parezca contraintuitivo, el P(cosine=good | hit) decrece con el aumento de K, pero es simple casualidad que se explica si nos fijamos en n_hit y n_miss. Estamos con un valor medio de P(cosine=good | hit) de 0.92, realmente alto. Es decir, el 92% de la veces que el modelo tenga el chunk necesario para responder, responderá. No obstante, hay que cogerlo con pinzas, pues un 92% sigue siendo demasiado alto.
    - El problema parece que viene cuando miramos P(cosine=good | miss). Vemos que empieza en 0.74 y acaba en 0.33. Valores realmente altos. Significa que, el 33% de las veces que NO tiene el contexto necesario para responder, responde bien. No obstante, si nos fijamos en el resultado de la celda de abajo, lo entenderemos:

In [106]:
MODEL = "gemma4_31b_4bit"
K = 5

hit_col = f"retrieval_hit@{K}"

df_miss_good = df_eval[
    (df_eval["model_alias"] == MODEL) &
    (df_eval[hit_col] == False) &
    (df_eval["cosine_decision"] == "good") &
    (~df_eval["is_hard_negative_eval"])  
]

print(f"MISS pero respuesta GOOD (sin hard negatives): {len(df_miss_good)}")

display(df_miss_good[[
    "question",
    "gold_answer",
    "generated_answer"
]].head(20))

MISS pero respuesta GOOD (sin hard negatives): 1


,question,gold_answer,generated_answer
276,¿Quién ataca a Ned en las calles de Desembarco del Rey tras la captura de Tyrion?,Jaime Lannister.,"thought\nNo lo sé con los fragmentos proporcionados.\n---\n**Explicación:**\nAl revisar los fragmentos, se observa que la captura de Tyrion es mencionada en el **Fragmento 3**, donde Ned se enfrenta a la Reina Cersei. Sin embargo, en ninguno de los textos se describe un ataque a Ned ""en las calles de Desembarco del Rey"" posterior a dicho evento. El"


Como vemos, el n_miss de Gemma-4 31B en top_5 es de 3. También vemos que ha acertado 1 NO teniendo contexto. Pero es que si nos fijamos en la respuesta, vemos que en realidad la respuesta está mal, aunque la similitud semántica la haya dado pos buena. Es decir, en realidad el modelo no está alucinando. No hace falta comentar los otros dos casos (con top_1 y top_3), pues ahí sí que tiene sentido que pueda responder bien sin estar el chunk correcto, pues todavía no hemos "visto" el chunk correcto.

Es decir, tanto en el modelo Gemma-4 31B, como en el resto de modelo pasa lo mismo. Después de haberlos inspeccionado de forma manual, el RAG está siendo capaz de responder siempre en base a los chunks recuperados. Si la información no está en los chunks, responderá mal, aunque la similitud semántica diga lo contrario.

Una vez explicado porque P(cosine=good | miss) es tan alto, sigamos con el resto de resultados.

- Resto de modelos:
    - Este es el siguiente modelo que mejores resultados ha obtenido. Con una probabilidad máxima del 73%, los resutlados no están nada mal. No obstante, no es capaz de hacerle sombra al modelo anterior. Como habíamos visto antes, el modelo falla o no sabe qué responder aunque el retrieval sea bueno.

    Esto queda demostrado comparando con el resultado de Gemma-4 31B, pues siendo el retrieval el mismo, este modelo falla casi un 20% más de preguntas. Lo mismo sucede con el resto de modelos.

Parece que no hay lugar a dudas de que tanto el retrieval como el modelo son buenos. No obstante, los modelos más pequeños no son capaces de responder de forma satisfactoria. 





### Capacidad de detectar Hard Negatives

Para terminar con la evaluación del Retrieval, veamos los resultados de las Hard Negatives. Esto sirve para ver si el modelo alucina o no. Si el modelo responde algo que no sea "No lo sé", está claro que está alucinando. Veamoslo:



In [107]:
# =========================
# FILTRAR HARD NEGATIVES
# =========================

df_hard = df_eval[df_eval["is_hard_negative_eval"]].copy()

print(f"Total hard negatives: {len(df_hard)}")

# =========================
# DISPLAY
# =========================

display(df_hard[[
    "idx",
    "question",
    "generated_answer",
    "cosine_decision"
]].sort_values("idx"))

Total hard negatives: 25


,idx,question,generated_answer,cosine_decision
100,101,¿Como muere sansa?,No lo sé con los fragmentos proporcionados.,good
205,101,¿Como muere sansa?,No lo sé con los fragmentos proporcionados.,good
415,101,¿Como muere sansa?,"¿Como muere sansa?\nRespuesta:\nNo lo sé con los fragmentos proporcionados.thought\nThe user is asking ""How does Sansa die?"" based on the provided texts.\nI need to check all fragments to see if there is any mention of Sansa's death.\n\nFragment 1 mentions: ""muerte, castillo, reino, odio"" and ""Sansa se ve obligada a asistir",borderline
310,101,¿Como muere sansa?,"thought\nNo lo sé con los fragmentos proporcionados.\n---\n**Explicación:** En los textos suministrados, Sansa no muere; se describe su situación como prisionera en el Torreón de Maegor y sus interacciones con Joffrey, pero no se menciona ningún fallecimiento del personaje.\n---\n**Nota:** Según las instrucciones, si la respuesta no se encuentra en los fragment",bad
520,101,¿Como muere sansa?,No lo sé con los fragmentos proporcionados.,good
206,102,¿Quien es el hijo bastardo de Jon Nieve?,No lo sé con los fragmentos proporcionados.,good
101,102,¿Quien es el hijo bastardo de Jon Nieve?,No lo sé con los fragmentos proporcionados.,good
311,102,¿Quien es el hijo bastardo de Jon Nieve?,thought\n---\n**No lo sé con los fragmentos proporcionados.**\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---\n---,good
416,102,¿Quien es el hijo bastardo de Jon Nieve?,No lo sé con los fragmentos proporcionados.end_of_turn><start_of_turn>model\nNo lo sé con los fragmentos proporcionados.</start_of_turn><end_of_turn><start_of_turn>model\nNo lo sé con los fragmentos proporcionados.</start_of_turn><end_of,good
521,102,¿Quien es el hijo bastardo de Jon Nieve?,No lo sé con los fragmentos proporcionados.,good


Si bien la salida del texto es un poco sucia, los resultados son maravillosos. En todos los casos, el modelo ha respondido que no lo sabe con los fragmentos proporcionados. 

Esto demuestra que el modelo de embedding es bueno, pues entiende que no hay forma de responder a esa pregunta, y también demuestra, aunque de forma mas indirecta, que el modelo no alucina e intenta responder aunque no sea. Es decir, el modelo se ciñe al contexto del RAG, no a nada que ya supiera de antemano, quitando el temor de antes en el que veiamos que P(cosine=good | miss) era alto a veces.

# Conclusiones finales

## Respuestas del RAG

Tras revisar en conjunto todos los bloques de evaluación (cosine+cross sobre respuesta gold, n-gramas y faithfulness), la conclusión fuerte es que **el ranking de modelos es estable**, pero también que **cada métrica cuenta una parte distinta de la historia**.

### 1) Rendimiento global por modelo (respuesta final)

- **Gemma-4 31B 4-bit** queda claramente primero: 93/105 aciertos (88.6%), con media coseno de 0.54. Es el modelo más fiable para este RAG, aunque también el más lento (~24 s de media).
- **Gemma-3 4B bf16** y **Gemma-3 4B 4-bit** forman el bloque intermedio (73% y ~69% de acierto, respectivamente): pierden precisión frente al 31B, pero ofrecen latencias bastante mejores.
- **Gemma-4 E4B 4-bit** y **Gemma-4 E4B bf16** quedan por detrás (~68% y ~65%): en este experimento, no trasladan a calidad final lo que cabría esperar por familia de modelo.

### 2) Qué nos dice cada método de evaluación

- **Cosine (gold vs respuesta)** funciona bien como métrica base para comparar modelos, pero por sí sola puede sobreestimar algunos casos “plausibles” semánticamente.
- **N-gramas / keyphrases** aporta señal léxica útil para inspección, pero mete ruido y no es suficientemente robusta como criterio principal (detecta incoherencias, pero también genera falsos positivos/negativos).
- **Faithfulness por cosine y por cross-encoder** deja claro que “respuesta parecida” no siempre significa “respuesta bien fundamentada en contexto”. El cross-encoder de faithfulness es más estricto y saca a la luz debilidades que el cosine no siempre captura.

### 3) Lectura integrada (lo importante de verdad)

- El 31B no solo gana en accuracy, también es el que mantiene mejor equilibrio entre precisión y consistencia entre métodos.
- En modelos pequeños, la degradación no es solo “fallar más”, sino también **ser más inestables según la métrica**.
- El análisis de discrepancias confirma que **no conviene decidir con una sola métrica**: para una evaluación seria de RAG hay que combinar señales semánticas, léxicas y de grounding.

### 4) Conclusión final de esta sección

El sistema RAG responde bien en términos generales y, con el modelo adecuado, alcanza un nivel alto de fiabilidad. Si el objetivo es calidad máxima, **Gemma-4 31B 4-bit** es la mejor opción del cuaderno; si se prioriza coste/latencia, las variantes 4B pueden servir, pero asumiendo una caída clara en robustez y precisión.


## Calidad del Retrieval

La evaluación de recuperación deja un mensaje muy positivo: **el retrieval está bien resuelto y no es el cuello de botella principal del sistema**.

### 1) Calidad intrínseca del retrieval

Sin hard negatives, las métricas son altas: recall@1 ≈ 72.6%, recall@3 ≈ 94.6%, recall@5 ≈ 96.6%, MRR ≈ 0.82. Esto indica que el chunk relevante suele aparecer muy arriba en el ranking.

Además, aunque el context precision (~0.48) pueda parecer moderado, es razonable para chunks largos (~6000 caracteres): no hace falta que todo el chunk sea relevante para sostener una buena respuesta.

### 2) Relación retrieval ↔ generación

El cruce P(cosine=good | hit) vs P(cosine=good | miss) requiere interpretación cuidadosa:

- Que exista P(good|miss) no implica necesariamente alucinación.
- En varios casos, el “miss” proviene de cómo está etiquetado el capítulo gold o de limitaciones de la métrica, no de que el modelo haya inventado contenido.
- La inspección manual del cuaderno respalda que, en general, las respuestas se apoyan en los fragmentos recuperados.

### 3) Lo que revela la comparación por modelos

Como el retrieval es común para todos, las diferencias grandes de calidad final entre modelos apuntan sobre todo a la **capacidad del generador para explotar contexto**:

- Con retrieval similar, el 31B convierte mejor ese contexto en respuesta correcta.
- Los modelos más pequeños fallan más incluso cuando el chunk útil sí está en top-k.

Es decir: en este pipeline, mejorar solo la recuperación ayuda, pero la mayor ganancia viene de elegir mejor modelo generativo o ajustar su inferencia/prompting.

### 4) Hard negatives y control de alucinación

El resultado en hard negatives es especialmente sólido: los modelos responden que no saben cuando no hay evidencia en contexto. Esto refuerza que el sistema está razonablemente bien alineado con el principio de grounding del RAG y no tiende a inventar por defecto.

### 5) Conclusión final de retrieval

El retrieval del proyecto es fuerte y consistente; el siguiente salto de calidad no pasa tanto por “recuperar mucho más”, sino por afinar la etapa generativa y mantener evaluación mixta (retrieval + calidad de respuesta + faithfulness + hard negatives) para no sacar conclusiones engañosas con una sola métrica.
